In [ ]:
import os
import gc
import xarray as xr
import numpy as np
import pandas as pd
from multiprocessing import Pool
from eofs.xarray import Eof
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ================= 配置区 =================
# 1. 原始数据目录
Z3_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002/Z3"
PS_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002/PS"

# 2. 数据输出目录
DATA_OUT_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002/NAM"
os.makedirs(DATA_OUT_DIR, exist_ok=True)

# 3. 图片输出目录
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# 4. 计算参数
LAT_MIN_AO = 20.0
LAT_MIN_NAM = 65.0
LAT_MAX_NAM = 90.0
TOTAL_YEARS = 210

# 这个插值计算明显比原来更重，建议先保守一点
CORES = 10

SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 104

# ------------------------------------------------------------
# 目标 pressure levels（hPa）
# ------------------------------------------------------------
TARGET_PLEV_HPA = np.array([
    1000.0, 950.0, 900.0, 850.0, 800.0, 750.0,
    700.0, 600.0, 550.0, 500.0, 450.0, 400.0,
    350.0, 300.0, 250.0, 225.0, 200.0, 175.0,
    150.0, 125.0, 100.0, 70.0, 50.0, 30.0,
    20.0, 10.0, 7.0, 5.0, 3.0, 2.0, 1.0
], dtype=np.float64)

TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0
AO_PLEV_HPA = 1000.0

# ------------------------------------------------------------
# 时间轴平移
# ------------------------------------------------------------
def shift_time_cftime(ds, year_offset):
    times = ds["time"].values
    new_times = [t.replace(year=t.year + year_offset) for t in times]
    ds["time"] = xr.DataArray(new_times, coords=[new_times], dims=["time"])
    return ds

# ------------------------------------------------------------
# 1D log-pressure 插值 + 边界线性外推
# 输入:
#   p_prof: 某个柱子的 pressure profile (Pa)
#   z_prof: 某个柱子的 Z3 profile
#   target_p: 目标 pressure levels (Pa)
# 输出:
#   对应 target_p 上的 Z3
# ------------------------------------------------------------
def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    # 按 pressure 从小到大排序，便于 np.interp 处理 log(p)
    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)

    # 先做区间内插值
    out = np.interp(ltp, lp, z)

    # 上边界外推（更高层，更小 pressure）
    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    # 下边界外推（更低层，更大 pressure，比如到 1000 hPa）
    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out

# ------------------------------------------------------------
# 将 Z3 从 hybrid 层插值/外推到目标 pressure levels
# 返回坐标名仍叫 lev，便于你下游脚本少改
# ------------------------------------------------------------
def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )

    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev["plev"].attrs["units"] = "hPa"
    z_on_plev["plev"].attrs["long_name"] = "pressure"

    # 调整维度顺序
    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")

    # 为了兼容你后面的代码和绘图，这里直接把 plev 改名为 lev
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    z_on_plev["lev"].attrs["units"] = "hPa"
    z_on_plev["lev"].attrs["long_name"] = "pressure"

    return z_on_plev

# ================= 1. 并行读取: 先统一插值到 pressure levels，再提取 AO 和 NAM =================
def process_one_year(year_int):
    f_z3 = os.path.join(Z3_DIR, f"B2000WCN.sample.cam.h3.{year_int:04d}.Z3.nc")
    f_ps = os.path.join(PS_DIR, f"B2000WCN.sample.cam.h3.{year_int:04d}.PS.nc")

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            # ------------------------------------------------------------
            # 先计算全层 pressure (Pa)
            # p = hyam * p0 + hybm * ps
            # ------------------------------------------------------------
            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]

            p3d = hyam * p0 + hybm * ps

            # 只保留 AO 需要的 20N-90N，减少计算量
            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, 90.0))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, 90.0))

            # ------------------------------------------------------------
            # 核心：全部剖面统一插值/外推到指定 pressure levels
            # 得到真实 pressure-coordinate 的 Z3
            # ------------------------------------------------------------
            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            # ------------------------------------------------------------
            # 任务 A: AO 用 1000 hPa
            # ------------------------------------------------------------
            da_z3_1000 = z3_plev_nh.sel(lev=AO_PLEV_HPA)
            da_z3_1000.name = "Z3_1000"

            # ------------------------------------------------------------
            # 任务 B: NAM 用全部 pressure levels 的 65-90N 极冠平均
            # ------------------------------------------------------------
            z3_polar = z3_plev_nh.sel(lat=slice(LAT_MIN_NAM, LAT_MAX_NAM))
            z3_pc_lonmean = z3_polar.mean("lon")
            weights = np.cos(np.deg2rad(z3_pc_lonmean["lat"]))
            da_z3_pc = z3_pc_lonmean.weighted(weights).mean("lat")
            da_z3_pc.name = "Z3_PC"

            # ------------------------------------------------------------
            # 时间轴修正
            # ------------------------------------------------------------
            if year_int >= 105:
                temp_ds1 = da_z3_1000.to_dataset(name="Z3_1000")
                temp_ds1 = shift_time_cftime(temp_ds1, SHIFT_RUN2_INTERNAL_YEAR_OFFSET)
                da_z3_1000 = temp_ds1["Z3_1000"]

                temp_ds2 = da_z3_pc.to_dataset(name="Z3_PC")
                temp_ds2 = shift_time_cftime(temp_ds2, SHIFT_RUN2_INTERNAL_YEAR_OFFSET)
                da_z3_pc = temp_ds2["Z3_PC"]

            out_1000 = da_z3_1000.compute()
            out_pc = da_z3_pc.compute()

            return (out_1000, out_pc)

    except Exception as e:
        print(f"Error in year {year_int:04d}: {str(e)}")
        return None


# ================= 主程序 =================
if __name__ == "__main__":
    print(f"🚀 开始使用 {CORES} 核并行提取 AO 与 NAM 基础数据...")
    years_to_process = list(range(1, TOTAL_YEARS + 1))

    with Pool(processes=CORES) as pool:
        results = pool.map(process_one_year, years_to_process)

    valid_results = [res for res in results if res is not None]

    # 解包
    z3_1000_all = xr.concat([res[0] for res in valid_results], dim="time").sortby("time")
    z3_pc_all = xr.concat([res[1] for res in valid_results], dim="time").sortby("time")
    del results, valid_results
    gc.collect()

    # =========================================================================
    #                                第一节：计算 AO
    # =========================================================================
    print("\n" + "=" * 50)
    print("🌟 第一节：处理地表 AO 模态与指数")
    print("=" * 50)

    # ---------------- AO monthly anomalies: 与 ERA5 版本保持一致 ----------------
    z3_1000_monthly = z3_1000_all.resample(time="1MS").mean().dropna(dim="time", how="all")
    clim_monthly = z3_1000_monthly.groupby("time.month").mean("time")
    monthly_anom = (z3_1000_monthly.groupby("time.month") - clim_monthly).drop_vars("month")

    coslat = np.clip(np.cos(np.deg2rad(monthly_anom.lat)), 0, None)
    wgts_2d = np.sqrt(coslat).broadcast_like(
        monthly_anom.isel(time=0).drop_vars("time", errors="ignore")
    )

    solver = Eof(monthly_anom, weights=wgts_2d.values)

    eof1 = solver.eofsAsCorrelation(neofs=1).squeeze()
    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze()

    if eof1_reg.sel(lat=slice(75, 90)).mean() > 0:
        eof1 = eof1 * -1
        eof1_reg = eof1_reg * -1
        pc1_raw = pc1_raw * -1
        flip_factor = -1
    else:
        flip_factor = 1

    # daily projection 需要 daily anomaly，这里补齐
    clim_daily = z3_1000_all.groupby("time.dayofyear").mean("time")
    clim_daily_smooth = clim_daily.rolling(dayofyear=21, center=True).mean().bfill("dayofyear").ffill("dayofyear")
    daily_anom = (z3_1000_all.groupby("time.dayofyear") - clim_daily_smooth).drop_vars("dayofyear")

    pseudo_pcs_raw = solver.projectField(daily_anom, neofs=1, eofscaling=0).squeeze() * flip_factor
    ao_index_daily = pseudo_pcs_raw / pc1_raw.std(dim="time")
    explained_var = solver.varianceFraction(neigs=1)[0].values * 100

    # ----------------- 绘图 -----------------
    from cartopy.util import add_cyclic_point

    # 图 1
    fig1 = plt.figure(figsize=(8, 8))
    ax1 = plt.axes(projection=ccrs.NorthPolarStereo())
    ax1.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
    ax1.coastlines(linewidth=1)
    ax1.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

    lon, lat = eof1.lon, eof1.lat
    eof1_cyclic, lon_cyclic = add_cyclic_point(eof1.values, coord=lon)

    cf1 = ax1.contourf(
        lon_cyclic, lat, eof1_cyclic,
        levels=np.linspace(-1, 1, 21),
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r",
        extend="both"
    )
    plt.colorbar(cf1, ax=ax1, orientation="horizontal", pad=0.05, aspect=40).set_label(
        "Correlation Coefficient", fontsize=12
    )
    plt.title(
        f"WACCM4 AO Spatial Pattern (EOF1)\nExplained Variance: {explained_var:.1f}%",
        fontsize=14, fontweight="bold", pad=15
    )
    fig_path1 = os.path.join(FIG_OUT_DIR, "WACCM4_AO_Pattern_Correlation.png")
    plt.savefig(fig_path1, dpi=300, bbox_inches="tight")

    # 图 2
    fig2 = plt.figure(figsize=(9, 9))
    ax2 = plt.axes(projection=ccrs.NorthPolarStereo())
    ax2.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
    ax2.coastlines(linewidth=1)
    ax2.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

    eof1_reg_cyclic, _ = add_cyclic_point(eof1_reg.values, coord=lon)

    noaa_colors = [
        "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5", "#a3dcfb", "#c2f0fb",
        "#dff8fd", "#f0fcfd", "#ffffff", "#fefbd9", "#feea9e", "#fec762",
        "#f27932", "#b82522"
    ]
    bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
    cmap2 = mcolors.ListedColormap(noaa_colors)
    norm2 = mcolors.BoundaryNorm(bounds, cmap2.N)

    cf2 = ax2.contourf(
        lon_cyclic, lat, eof1_reg_cyclic,
        levels=bounds,
        transform=ccrs.PlateCarree(),
        cmap=cmap2,
        norm=norm2,
        extend="both"
    )
    cbar2 = plt.colorbar(
        cf2, ax=ax2, orientation="vertical", shrink=0.7, pad=0.05,
        ticks=[-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
    )
    cbar2.ax.tick_params(labelsize=10)
    plt.title(
        f"Leading EOF ({explained_var:.0f}%) shown as\nregression map of 1000mb height (m)",
        fontsize=16, fontname="Comic Sans MS", pad=20
    )
    fig_path2 = os.path.join(FIG_OUT_DIR, "WACCM4_AO_Regression_NOAA_Style.png")
    plt.savefig(fig_path2, dpi=300, bbox_inches="tight")

    # ----------------- 数据输出 -----------------
    csv_path = os.path.join(DATA_OUT_DIR, "WACCM4_Daily_AO_Index.csv")
    pd.DataFrame({
        "Date": ao_index_daily.time.values,
        "AO_Index": ao_index_daily.values
    }).to_csv(csv_path, index=False)

    eof1_reg.name = "EOF1_Regression"
    eof1_reg.to_netcdf(os.path.join(DATA_OUT_DIR, "WACCM4_EOF1_Regression.nc"))

    eof1.name = "EOF1_Correlation"
    eof1.to_netcdf(os.path.join(DATA_OUT_DIR, "WACCM4_EOF1_Correlation.nc"))

    print(f"✅ AO 数据已成功归档至: {DATA_OUT_DIR}")
    print(f"✅ AO 图表已成功保存至: {FIG_OUT_DIR}")

    # =========================================================================
    #                                第二节：计算 NAM
    # =========================================================================
    print("\n" + "=" * 50)
    print("🌟 第二节：计算多层垂直 NAM 指数")
    print("=" * 50)

    print("📊 正在处理极冠高度异常场...")

    # 这里的 lev 现在已经是真正的 pressure [hPa]
    # 如果你想进一步贴近 ERA5 那版，可以去掉 21 天平滑
    clim_pc = z3_pc_all.groupby("time.dayofyear").mean("time")
    anom_pc = (z3_pc_all.groupby("time.dayofyear") - clim_pc).drop_vars("dayofyear")

    print("🌀 正在进行 NAM 标准化计算...")
    std_pc = anom_pc.std("time")
    nam_vertical = -(anom_pc / std_pc)

    nam_vertical.name = "NAM_Vertical"
    nam_vertical.attrs["long_name"] = "Vertical NAM index from standardized 65-90N geopotential height anomalies on pressure levels"
    nam_vertical.attrs["description"] = "Computed after log-pressure interpolation/extrapolation to fixed pressure levels and standardized by full-period std, multiplied by -1."
    nam_vertical["lev"].attrs["units"] = "hPa"
    nam_vertical["lev"].attrs["long_name"] = "pressure"

    # ----------------- 数据输出 -----------------
    nc_nam_path = os.path.join(DATA_OUT_DIR, "WACCM4_Vertical_NAM_210yrs.nc")
    nam_vertical.to_netcdf(nc_nam_path)
    print(f"✅ 210年全量多层垂直 NAM 已保存至: {nc_nam_path}")

    # ================= 结束 =================
    plt.show()
    print("\n🎉 全部计算与归档任务圆满完成！")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

# ================= 配置区 =================
CSV_PATH = '/home/weiji/restart_exam/code/20260315NAM/resul/WACCM4_Daily_AO_Index.csv'
OUT_FIG = '/home/weiji/restart_exam/code/20260315NAM/resul/WACCM4_AO_IntraAnnual.png'

print("📂 正在读取每日 AO 指数数据...")
df = pd.read_csv(CSV_PATH)

print("⏳ 正在处理时间轴并计算 DOY 统计量...")
# 1. 将 Date 列转为字符串，提取月和日（假设格式为 "YYYY-MM-DD HH:MM:SS"）
df['Date_str'] = df['Date'].astype(str)
df['Month'] = df['Date_str'].str[5:7].astype(int)
df['Day'] = df['Date_str'].str[8:10].astype(int)

# 2. 构造一个用于绘图的“伪日期”（使用 2023 这样的平年，避开 0001 年的 Pandas 限制）
# coerce 模式会把由于闰年多出来的 2月29日 变成 NaT，对于 365 天日历正好忽略
df['Fake_Date'] = pd.to_datetime('2023-' + df['Month'].astype(str).str.zfill(2) + '-' + df['Day'].astype(str).str.zfill(2), errors='coerce')

# 3. 按 DOY (用 Fake_Date 代理) 计算 210 年间的平均值和标准差
daily_stats = df.groupby('Fake_Date')['AO_Index'].agg(['mean', 'std']).reset_index()

print("📈 正在绘制年中变率折线图...")
# ================= 绘图 =================
fig, ax = plt.subplots(figsize=(10, 6))

# 绘制 ±1 个标准差的阴影区，代表每一天的气候变率范围 (Variability)
ax.fill_between(daily_stats['Fake_Date'], 
                daily_stats['mean'] - daily_stats['std'], 
                daily_stats['mean'] + daily_stats['std'], 
                color='#1f77b4', alpha=0.3, edgecolor='none', 
                label='±1 Std Dev (Interannual Variability)')

# 绘制 210 年平均的日气候态折线
ax.plot(daily_stats['Fake_Date'], daily_stats['mean'], 
        color='black', linewidth=2, label='Daily Mean AO Index')

# 添加 0 刻度基准线
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.8)

# ================= 坐标轴美化 =================
# 将 X 轴格式化为月份缩写 (Jan, Feb, Mar...)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

ax.set_title('Intra-annual Variability of WACCM4 AO Index (210-Year Climatology)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month of Year', fontsize=13)
ax.set_ylabel('Standardized AO Index', fontsize=13)

ax.legend(loc='upper right', frameon=True, fontsize=11)
ax.grid(True, linestyle=':', alpha=0.7)

# 设置 X 轴的显示范围为 1月1日 到 12月31日
ax.set_xlim([pd.to_datetime('2023-01-01'), pd.to_datetime('2023-12-31')])

plt.savefig(OUT_FIG, dpi=300, bbox_inches='tight')
print(f"🖼️ 图表已保存至: {OUT_FIG}")

plt.show()

In [ ]:
import os
import gc
import glob
import xarray as xr
import numpy as np
import pandas as pd
from multiprocessing import Pool
from eofs.xarray import Eof
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ================= 1. 核心配置区 =================
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

LAT_MIN_AO = 20.0
LAT_MIN_NAM = 65.0
LAT_MAX_NAM = 90.0

# ---------------- 新增：目标 pressure levels ----------------
TARGET_PLEV_HPA = np.array([
    1000.0, 950.0, 900.0, 850.0, 800.0, 750.0,
    700.0, 600.0, 550.0, 500.0, 450.0, 400.0,
    350.0, 300.0, 250.0, 225.0, 200.0, 175.0,
    150.0, 125.0, 100.0, 70.0, 50.0, 30.0,
    20.0, 10.0, 7.0, 5.0, 3.0, 2.0, 1.0
], dtype=np.float64)

TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0
AO_PLEV_HPA = 1000.0

# ---------------- 建议先保守一点，插值比以前更重 ----------------
CORES = 10
SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 104

# ================= 【核心修改】：加入文件命名模板 =================
EXPERIMENTS = [
    {
        "name": "B2000WCN001002",
        "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN001002",
        "file_template": "B2000WCN.sample.cam.h3.{year:04d}.{var}.nc",
        "apply_shift": True,
        "is_reference": True,
        "use_reference": False
    },
    {
        "name": "B2000WCN_NOCOUPL001002",
        "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002",
        "file_template": "B2000WCN.NOCOUPL.sample.cam.h3.{year:04d}.{var}.nc",
        "apply_shift": True,
        "is_reference": False,
        "use_reference": False
    },
    {
        "name": "BWCN",
        "base_dir": "/mnt/soclim0/public_data/weiji/BWCN",
        "file_template": "BWCN.cam.h3.{year:04d}.{var}.nc",
        "apply_shift": False,
        "is_reference": False,
        "use_reference": True
    }
]

REF_DATA = {}

# ================= 2. 基础工具函数 =================
def shift_time_cftime(ds, year_offset):
    times = ds["time"].values
    new_times = [t.replace(year=t.year + year_offset) for t in times]
    ds["time"] = xr.DataArray(new_times, coords=[new_times], dims=["time"])
    return ds

def _process_wrapper(args):
    return process_one_year(*args)

# ------------------------------------------------------------
# 1D log-pressure 插值 + 边界线性外推
# p_prof: Pa
# z_prof: geopotential height
# target_p: Pa
# ------------------------------------------------------------
def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    # pressure 从小到大排序，便于 np.interp 在 log(p) 空间处理
    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)

    out = np.interp(ltp, lp, z)

    # 向更高层外推（更小 pressure）
    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    # 向近地层外推（更大 pressure，例如到 1000 hPa）
    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out

# ------------------------------------------------------------
# 将 Z3 从 hybrid/model lev 插值到固定 pressure levels
# 输出坐标名仍叫 lev，便于你后面 AO/NAM 代码尽量少改
# ------------------------------------------------------------
def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )

    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev["plev"].attrs["units"] = "hPa"
    z_on_plev["plev"].attrs["long_name"] = "pressure"

    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    z_on_plev["lev"].attrs["units"] = "hPa"
    z_on_plev["lev"].attrs["long_name"] = "pressure"

    return z_on_plev

def process_one_year(year_int, base_dir, apply_shift, file_template):
    # 利用模板动态生成对应实验的文件名
    f_z3_name = file_template.format(year=year_int, var="Z3")
    f_ps_name = file_template.format(year=year_int, var="PS")

    f_z3 = os.path.join(base_dir, "Z3", f_z3_name)
    if not os.path.exists(f_z3):
        f_z3 = os.path.join(base_dir, f_z3_name)

    f_ps = os.path.join(base_dir, "PS", f_ps_name)
    if not os.path.exists(f_ps):
        f_ps = os.path.join(base_dir, f_ps_name)

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            # ------------------------------------------------------------
            # 先计算全层 pressure (Pa)
            # ------------------------------------------------------------
            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]

            p3d = hyam * p0 + hybm * ps

            # 只保留北半球 20N-90N，减少计算量
            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, 90.0))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, 90.0))

            # ------------------------------------------------------------
            # 核心：先统一外推+插值到固定 pressure levels
            # ------------------------------------------------------------
            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            # --- A: AO 用 1000 hPa ---
            da_z3_1000 = z3_plev_nh.sel(lev=AO_PLEV_HPA)
            da_z3_1000.name = "Z3_1000"

            # --- B: NAM 用固定 pressure levels 上的极冠平均 ---
            z3_polar = z3_plev_nh.sel(lat=slice(LAT_MIN_NAM, LAT_MAX_NAM))
            z3_pc_lonmean = z3_polar.mean("lon")
            weights = np.cos(np.deg2rad(z3_pc_lonmean["lat"]))
            da_z3_pc = z3_pc_lonmean.weighted(weights).mean("lat")
            da_z3_pc.name = "Z3_PC"

            # --- 时间修正 ---
            if apply_shift and year_int >= 105:
                da_z3_1000 = shift_time_cftime(
                    da_z3_1000.to_dataset(name="Z3_1000"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z3_1000"]

                da_z3_pc = shift_time_cftime(
                    da_z3_pc.to_dataset(name="Z3_PC"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z3_PC"]

            out_1000 = da_z3_1000.compute()
            out_1000.name = "Z3_1000"

            out_pc = da_z3_pc.compute()
            out_pc.name = "Z3_PC"
            out_pc["lev"].attrs["units"] = "hPa"
            out_pc["lev"].attrs["long_name"] = "pressure"

            return (out_1000, out_pc)

    except Exception as e:
        print(f"Error in {base_dir} year {year_int:04d}: {str(e)}")
        return None

# ================= 3. 主循环流水线 =================
if __name__ == "__main__":
    for exp in EXPERIMENTS:
        exp_name = exp["name"]
        base_dir = exp["base_dir"]
        apply_shift = exp["apply_shift"]
        file_template = exp["file_template"]

        data_out_dir = os.path.join(base_dir, "NAM")
        os.makedirs(data_out_dir, exist_ok=True)

        print("\n" + "🚀" * 25)
        print(f"正在处理实验组: {exp_name}")
        print("🚀" * 25)

        z3_files = glob.glob(os.path.join(base_dir, "Z3", "*.nc"))
        if not z3_files:
            z3_files = glob.glob(os.path.join(base_dir, "*.Z3.nc"))

        max_year = len(z3_files) if len(z3_files) > 0 else 250
        years_to_process = list(range(1, max_year + 1))

        print(f"--> 预计扫描 {len(years_to_process)} 年的数据...")
        args_list = [(y, base_dir, apply_shift, file_template) for y in years_to_process]

        with Pool(processes=CORES) as pool:
            results = pool.map(_process_wrapper, args_list)

        valid_results = [res for res in results if res is not None]
        if not valid_results:
            print(f"❌ 警告：未找到 {exp_name} 的有效数据，跳过该组。")
            continue

        z3_1000_all = xr.concat([res[0] for res in valid_results], dim="time").sortby("time")
        z3_pc_all = xr.concat([res[1] for res in valid_results], dim="time").sortby("time")
        z3_pc_all["lev"].attrs["units"] = "hPa"
        z3_pc_all["lev"].attrs["long_name"] = "pressure"
        del results, valid_results
        gc.collect()

        # =========================================================================
        # 第一节：计算 AO
        # =========================================================================
        print(f"🌟 开始处理 {exp_name} 的 AO 指数...")

        if not exp["use_reference"]:
            clim_1000 = z3_1000_all.groupby("time.dayofyear").mean("time")
            clim_1000_smooth = clim_1000.rolling(dayofyear=21, center=True).mean().bfill("dayofyear").ffill("dayofyear")
            daily_anom = (z3_1000_all.groupby("time.dayofyear") - clim_1000_smooth).drop_vars("dayofyear")

            monthly_anom = daily_anom.resample(time="1MS").mean().dropna(dim="time", how="all")
            coslat = np.clip(np.cos(np.deg2rad(monthly_anom.lat)), 0, None)
            wgts_2d = np.sqrt(coslat).broadcast_like(
                monthly_anom.isel(time=0).drop_vars("time", errors="ignore")
            )

            solver = Eof(monthly_anom, weights=wgts_2d.values)
            eof1 = solver.eofsAsCorrelation(neofs=1).squeeze()
            eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
            pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze()

            if eof1_reg.sel(lat=slice(75, 90)).mean() > 0:
                eof1 = eof1 * -1
                eof1_reg = eof1_reg * -1
                pc1_raw = pc1_raw * -1
                flip_factor = -1
            else:
                flip_factor = 1

            sigma_monthly = pc1_raw.std(dim="time")

            if exp["is_reference"]:
                REF_DATA["clim_1000_smooth"] = clim_1000_smooth
                REF_DATA["solver"] = solver
                REF_DATA["flip_factor"] = flip_factor
                REF_DATA["sigma_monthly"] = sigma_monthly
                print(f"--> [记录] {exp_name} 的基准 AO 气候态与求解器已缓存。")

            explained_var = solver.varianceFraction(neigs=1)[0].values * 100

            # ------- 绘制专属空间图 -------
            from cartopy.util import add_cyclic_point

            lon, lat = eof1.lon, eof1.lat
            eof1_cyclic, lon_cyclic = add_cyclic_point(eof1.values, coord=lon)

            fig1 = plt.figure(figsize=(8, 8))
            ax1 = plt.axes(projection=ccrs.NorthPolarStereo())
            ax1.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
            ax1.coastlines(linewidth=1)
            ax1.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)
            cf1 = ax1.contourf(
                lon_cyclic, lat, eof1_cyclic,
                levels=np.linspace(-1, 1, 21),
                transform=ccrs.PlateCarree(),
                cmap="RdBu_r",
                extend="both"
            )
            plt.colorbar(cf1, ax=ax1, orientation="horizontal", pad=0.05, aspect=40).set_label(
                "Correlation Coefficient", fontsize=12
            )
            plt.title(
                f"{exp_name} AO Spatial Pattern\nExplained Variance: {explained_var:.1f}%",
                fontsize=14, fontweight="bold", pad=15
            )
            plt.savefig(os.path.join(FIG_OUT_DIR, f"{exp_name}_AO_Pattern_Correlation.png"), dpi=300, bbox_inches="tight")
            plt.close(fig1)

            eof1_reg_cyclic, _ = add_cyclic_point(eof1_reg.values, coord=lon)
            fig2 = plt.figure(figsize=(9, 9))
            ax2 = plt.axes(projection=ccrs.NorthPolarStereo())
            ax2.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
            ax2.coastlines(linewidth=1)
            ax2.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

            noaa_colors = [
                "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5", "#a3dcfb", "#c2f0fb",
                "#dff8fd", "#f0fcfd", "#ffffff", "#fefbd9", "#feea9e", "#fec762",
                "#f27932", "#b82522"
            ]
            bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
            cmap2 = mcolors.ListedColormap(noaa_colors)
            norm2 = mcolors.BoundaryNorm(bounds, cmap2.N)

            cf2 = ax2.contourf(
                lon_cyclic, lat, eof1_reg_cyclic,
                levels=bounds,
                transform=ccrs.PlateCarree(),
                cmap=cmap2,
                norm=norm2,
                extend="both"
            )
            cbar2 = plt.colorbar(
                cf2, ax=ax2, orientation="vertical", shrink=0.7, pad=0.05,
                ticks=[-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
            )
            cbar2.ax.tick_params(labelsize=10)
            plt.title(
                f"{exp_name} Leading EOF ({explained_var:.0f}%) shown as\nregression map of 1000mb height (m)",
                fontsize=16, pad=20
            )
            plt.savefig(os.path.join(FIG_OUT_DIR, f"{exp_name}_AO_Regression_NOAA_Style.png"), dpi=300, bbox_inches="tight")
            plt.close(fig2)

            eof1_reg.name = "EOF1_Regression"
            eof1_reg.to_netcdf(os.path.join(data_out_dir, f"{exp_name}_EOF1_Regression.nc"))

            eof1.name = "EOF1_Correlation"
            eof1.to_netcdf(os.path.join(data_out_dir, f"{exp_name}_EOF1_Correlation.nc"))

        else:
            print(f"--> [核心] 正在应用基准组 (Couple) 的气候态与 EOF 求解器...")
            ref_clim = REF_DATA["clim_1000_smooth"]
            ref_solver = REF_DATA["solver"]
            flip_factor = REF_DATA["flip_factor"]
            sigma_monthly = REF_DATA["sigma_monthly"]

            daily_anom = (z3_1000_all.groupby("time.dayofyear") - ref_clim).drop_vars("dayofyear")
            solver = ref_solver
            print("--> (BWCN 组借用参考组模态，跳过绘制专属 EOF 空间图)")

        pseudo_pcs_raw = solver.projectField(daily_anom, neofs=1, eofscaling=0).squeeze() * flip_factor
        ao_index_daily = pseudo_pcs_raw / sigma_monthly

        csv_path = os.path.join(data_out_dir, f"{exp_name}_Daily_AO_Index.csv")
        pd.DataFrame({
            "Date": ao_index_daily.time.values,
            "AO_Index": ao_index_daily.values
        }).to_csv(csv_path, index=False)
        print(f"✅ {exp_name} AO 数据已成功归档至: {data_out_dir}")

        # =========================================================================
        # 第二节：计算 NAM
        # =========================================================================
        print(f"🌟 开始处理 {exp_name} 的多层垂直 NAM 指数...")

        if not exp["use_reference"]:
            clim_pc = z3_pc_all.groupby("time.dayofyear").mean("time")
            clim_pc_smooth = clim_pc.rolling(dayofyear=21, center=True).mean().bfill("dayofyear").ffill("dayofyear")
            anom_pc = (z3_pc_all.groupby("time.dayofyear") - clim_pc_smooth).drop_vars("dayofyear")
            std_pc = anom_pc.std("time")

            if exp["is_reference"]:
                REF_DATA["clim_pc_smooth"] = clim_pc_smooth
                REF_DATA["std_pc"] = std_pc
                print(f"--> [记录] {exp_name} 的基准 NAM 气候态与标准差已缓存。")
        else:
            print(f"--> [核心] 正在应用基准组的 NAM 气候态与标准差...")
            ref_clim_pc = REF_DATA["clim_pc_smooth"]
            std_pc = REF_DATA["std_pc"]
            anom_pc = (z3_pc_all.groupby("time.dayofyear") - ref_clim_pc).drop_vars("dayofyear")

        nam_vertical = -(anom_pc / std_pc)
        nam_vertical.name = "NAM_Vertical"
        nam_vertical.attrs["long_name"] = f"Vertical NAM index ({exp_name}) on pressure levels"
        nam_vertical["lev"].attrs["units"] = "hPa"
        nam_vertical["lev"].attrs["long_name"] = "pressure"

        nc_nam_path = os.path.join(data_out_dir, f"{exp_name}_Vertical_NAM.nc")
        nam_vertical.to_netcdf(nc_nam_path)
        print(f"✅ {exp_name} 全量多层垂直 NAM 已保存至: {nc_nam_path}")

        del z3_1000_all, z3_pc_all, daily_anom, anom_pc, nam_vertical
        gc.collect()

    print("\n🎉 全部 3 组跨实验对比计算与归档任务圆满完成！")

In [ ]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================= 1. 核心配置区 =================
# 实验组名称
EXP_NAME = "BWCN"

# 数据输入目录 (指向我们刚刚算好并保存的 NAM 文件夹)
DATA_DIR = f"/mnt/soclim0/public_data/weiji/{EXP_NAME}/NAM"

# 图片输出目录
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# 输入文件完整路径
NAM_FILE = os.path.join(DATA_DIR, f"{EXP_NAME}_Vertical_NAM.nc")
AO_FILE = os.path.join(DATA_DIR, f"{EXP_NAME}_Daily_AO_Index.csv")

# 绘图的时间窗口 (模拟参考图中横跨整个冬季的视图)
START_DATE = "0007-11-01"
END_DATE = "0008-05-31"

print(f"📂 正在读取 {EXP_NAME} 组的 NAM 和 AO 数据...")

# ================= 2. 数据读取与精准截取 =================
# 读取多层垂直 NAM 数据
ds_nam = xr.open_dataset(NAM_FILE)
nam_da = ds_nam["NAM_Vertical"]

# 利用 xarray 的 slice 完美截取时间段
nam_sub = nam_da.sel(time=slice(START_DATE, END_DATE))

# 读取地面 AO CSV 数据
ao_df = pd.read_csv(AO_FILE)

# 确保 CSV 中的日期格式干净，并按字符串切片出相同的日期范围
ao_df['Date_str'] = ao_df['Date'].astype(str).str[:10]
mask = (ao_df['Date_str'] >= START_DATE) & (ao_df['Date_str'] <= END_DATE)
ao_sub = ao_df[mask].copy()

# 提取用于绘图的数组
time_arr = nam_sub.time.values
ao_vals = ao_sub['AO_Index'].values

print("🎨 正在渲染平流层-对流层耦合 (Time-Height) 剖面图...")

# ================= 3. 高级布局与绘图 (解决严丝合缝对齐问题) =================
# 【调整高度 1】：总画布变矮一点，显得更修长 (10 为宽，6 为高)
fig = plt.figure(figsize=(10, 6))

# 【核心布局】：将网格分为 2 行 2 列。
# height_ratios=[2.0, 1.0] 控制上下比例，图 a 变矮了 (2:1)
# width_ratios=[1, 0.02] 控制左右比例，右边留出 2% 的极窄宽度专门放色标！
gs = fig.add_gridspec(nrows=2, ncols=2, 
                      height_ratios=[2.0, 1.0], 
                      width_ratios=[1, 0.02], 
                      hspace=0.08, wspace=0.02)

# -------------------------------------------------------------
# 面板 (a): NAM Time-Height 等值线图 (放在左上角 [0, 0])
# -------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])

# 专门给 Colorbar 留的独立房间 (放在右上角 [0, 1])
cax = fig.add_subplot(gs[0, 1]) 

levels_c = np.arange(-4.5, 5.0, 0.5)

cf = nam_sub.plot.contourf(
    ax=ax1,
    x="time",
    y="lev",
    levels=levels_c,
    cmap="RdBu",   
    extend="both",
    add_colorbar=False # 关闭自带色标，防止它挤压主图宽度
)

ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_ylim(1000, 1) 
ax1.set_yticks([1000, 300, 100, 30, 10, 3, 1])
ax1.get_yaxis().set_major_formatter(plt.ScalarFormatter())

ax1.set_ylabel("Pressure [hPa]", fontsize=16)
ax1.set_xlabel("")
ax1.tick_params(labelbottom=False)  
ax1.set_title("") 
ax1.text(0.01, 0.92, "(a)", transform=ax1.transAxes, fontsize=16, fontweight="bold",
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))

# 将 Colorbar 画在我们专门预留的 cax 房间里，彻底解决挤压对齐问题！
cbar = fig.colorbar(cf, cax=cax)
cbar.set_label("NAM Index", fontsize=14)

# -------------------------------------------------------------
# 面板 (b): 地面 AO 时间序列折线图 (放在左下角 [1, 0])
# -------------------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)

ax2.plot(time_arr, ao_vals, color='black', lw=1.8)
ax2.axhline(0, color='gray', lw=1.2, linestyle='-')

ax2.set_ylabel(r"$AO_{\mathrm{WACCM}}$", fontsize=16)
ax2.set_ylim(-4.5, 4.5)
ax2.set_yticks([-3, 0, 3])
ax2.text(0.01, 0.82, "(b)", transform=ax2.transAxes, fontsize=16, fontweight="bold",
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
ax2.grid(True, linestyle=':', alpha=0.6)

# ================= 4. 标题与保存 =================
fig.suptitle(f'Time series of NAM and AO indices ({EXP_NAME}: Yr 0007-0008)', 
             fontsize=18, fontweight='bold', y=0.95)

out_fig = os.path.join(FIG_OUT_DIR, f"{EXP_NAME}_NAM_AO_TimeHeight_Yr0007_0008.png")
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
print(f"✅ 绝美且绝对对齐的剖面图已保存至: {out_fig}")

plt.show()

# 及时关闭数据集释放内存
ds_nam.close()

In [ ]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================= 1. 路径配置 =================
NAM_FILE = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence/ERA5_daily_vertical_NAM_20191101_20200531.nc"
AO_FILE  = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence/AO_CPC_20191101_20200531.csv"

FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

OUT_FIG = os.path.join(FIG_OUT_DIR, "Figure3ab_ERA5NAM_AOCPC_201911_202005_replot.png")

START_DATE = "2019-11-01"
END_DATE   = "2020-05-01"

print("📂 正在读取已经保存好的 NAM 和 AO 数据...")

# ================= 2. 读取数据 =================
# ---- NAM ----
ds_nam = xr.open_dataset(NAM_FILE)
nam_da = ds_nam["NAM"]   # 你的变量名就是 NAM

# 再保险切一次时间
nam_sub = nam_da.sel(time=slice(START_DATE, END_DATE))

# ---- AO ----
ao_df = pd.read_csv(AO_FILE, parse_dates=["time"])
ao_sub = ao_df[(ao_df["time"] >= START_DATE) & (ao_df["time"] <= END_DATE)].copy()

# 提取数组
time_arr = nam_sub["time"].values
ao_time  = ao_sub["time"].values
ao_vals  = ao_sub["ao"].values

print("🎨 正在按新风格重绘 Figure 3a,b ...")

# ================= 3. 新风格绘图 =================
fig = plt.figure(figsize=(10, 6))

gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[2.0, 1.0],
    width_ratios=[1, 0.02],
    hspace=0.08, wspace=0.02
)

# -------------------------------------------------
# Panel (a): NAM time-height
# -------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

levels_c = np.arange(-4.5, 5.0, 0.5)

cf = nam_sub.plot.contourf(
    ax=ax1,
    x="time",
    y="level",            # 你的坐标名是 level
    levels=levels_c,
    cmap="RdBu",
    extend="both",
    add_colorbar=False
)

ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_ylim(1000, 1)
ax1.set_yticks([1000, 300, 100, 30, 10, 3, 1])
ax1.get_yaxis().set_major_formatter(plt.ScalarFormatter())

ax1.set_ylabel("Pressure [hPa]", fontsize=16)
ax1.set_xlabel("")
ax1.tick_params(labelbottom=False)
ax1.set_title("")
ax1.text(
    0.01, 0.92, "(a)",
    transform=ax1.transAxes,
    fontsize=16, fontweight="bold",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2)
)

cbar = fig.colorbar(cf, cax=cax)
cbar.set_label("NAM Index", fontsize=14)

# -------------------------------------------------
# Panel (b): AO_CPC
# -------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)

ax2.plot(ao_time, ao_vals, color="black", lw=1.8)
ax2.axhline(0, color="gray", lw=1.2, linestyle="-")

ax2.set_ylabel(r"$AO_{\mathrm{CPC}}$", fontsize=16)
ax2.set_ylim(-4.5, 4.5)
ax2.set_yticks([-3, 0, 3])
ax2.text(
    0.01, 0.82, "(b)",
    transform=ax2.transAxes,
    fontsize=16, fontweight="bold",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2)
)
ax2.grid(True, linestyle=":", alpha=0.6)

# ================= 4. 标题与保存 =================
fig.suptitle(
    "Time series of ERA5 vertical NAM and CPC AO indices (2019/11–2020/05)",
    fontsize=18, fontweight="bold", y=0.95
)

plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
print(f"✅ 已保存到: {OUT_FIG}")

plt.show()
ds_nam.close()

In [ ]:
import os
import re
import gc
import glob
import xarray as xr
import numpy as np
import pandas as pd
from multiprocessing import Pool
from eofs.xarray import Eof

# ============================================================
# 1. 核心配置区
# ============================================================
HINDCAST_ROOT = "/mnt/soclim0/public_data/weiji/Hindcast"

# 参考气候态：始终使用 B2000WCN001002
REF_EXP = {
    "name": "B2000WCN001002",
    "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN001002",
    "file_template": "B2000WCN.sample.cam.h3.{year:04d}.{var}.nc",
    "apply_shift": True,
}

LAT_MIN_AO = 20.0
LAT_MIN_NAM = 65.0
LAT_MAX_NAM = 90.0

# ---------------- 新增：固定 pressure levels ----------------
TARGET_PLEV_HPA = np.array([
    1000.0, 950.0, 900.0, 850.0, 800.0, 750.0,
    700.0, 600.0, 550.0, 500.0, 450.0, 400.0,
    350.0, 300.0, 250.0, 225.0, 200.0, 175.0,
    150.0, 125.0, 100.0, 70.0, 50.0, 30.0,
    20.0, 10.0, 7.0, 5.0, 3.0, 2.0, 1.0
], dtype=np.float64)

TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0
AO_PLEV_HPA = 1000.0

# 插值比原来更吃资源，建议先别开太多核
CORES = 10
SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 104

# True = 连 *_NOCOUPL 目录也处理
# False = 只处理 BWCN 来源 hindcast（推荐，符合你当前需求）
INCLUDE_NOCOUPL_HINDCAST = False

REF_DATA = {}


# ============================================================
# 2. 基础工具函数
# ============================================================
def shift_time_cftime(ds, year_offset):
    times = ds["time"].values
    new_times = [t.replace(year=t.year + year_offset) for t in times]
    ds["time"] = xr.DataArray(new_times, coords=[new_times], dims=["time"])
    return ds


def drop_coord_if_present(da, coord_name):
    if coord_name in da.coords:
        return da.drop_vars(coord_name)
    return da


def _process_ref_wrapper(args):
    return process_reference_one_year(*args)


def _process_hindcast_wrapper(args):
    return process_one_hindcast_file(*args)


# ============================================================
# 3. 垂直插值工具：log-pressure 插值 + 边界线性外推
# ============================================================
def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    """
    p_prof : 1D pressure profile, Pa
    z_prof : 1D geopotential height profile
    target_p: target pressure levels, Pa
    """
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    # pressure 从小到大排序，便于在 log(p) 空间插值
    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)

    # 区间内插值
    out = np.interp(ltp, lp, z)

    # 向更高层外推（更小 pressure）
    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    # 向更近地层外推（更大 pressure，例如到 1000 hPa）
    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out


def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    """
    将 Z3 从原始 hybrid/model lev 插值到固定 pressure levels。
    输出坐标名仍然叫 lev，方便下游代码少改。
    """
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )

    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev["plev"].attrs["units"] = "hPa"
    z_on_plev["plev"].attrs["long_name"] = "pressure"

    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    z_on_plev["lev"].attrs["units"] = "hPa"
    z_on_plev["lev"].attrs["long_name"] = "pressure"

    return z_on_plev


# ============================================================
# 4. 单年参考资料处理：B2000WCN001002
# ============================================================
def process_reference_one_year(year_int, base_dir, apply_shift, file_template):
    f_z3_name = file_template.format(year=year_int, var="Z3")
    f_ps_name = file_template.format(year=year_int, var="PS")

    f_z3 = os.path.join(base_dir, "Z3", f_z3_name)
    if not os.path.exists(f_z3):
        f_z3 = os.path.join(base_dir, f_z3_name)

    f_ps = os.path.join(base_dir, "PS", f_ps_name)
    if not os.path.exists(f_ps):
        f_ps = os.path.join(base_dir, f_ps_name)

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            # ------------------------------------------------------------
            # 先构造全层 pressure (Pa)
            # ------------------------------------------------------------
            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]

            p3d = hyam * p0 + hybm * ps

            # 只保留 20N-90N，减少计算量
            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, 90.0))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, 90.0))

            # ------------------------------------------------------------
            # 核心：先插值/外推到固定 pressure levels
            # ------------------------------------------------------------
            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            # --- A. AO 用 1000 hPa ---
            da_z3_1000 = z3_plev_nh.sel(lev=AO_PLEV_HPA)
            da_z3_1000.name = "Z3_1000"

            # --- B. NAM 用固定 pressure levels 上的极冠平均 ---
            z3_polar = z3_plev_nh.sel(lat=slice(LAT_MIN_NAM, LAT_MAX_NAM))
            z3_pc_lonmean = z3_polar.mean("lon")
            weights = np.cos(np.deg2rad(z3_pc_lonmean["lat"]))
            da_z3_pc = z3_pc_lonmean.weighted(weights).mean("lat")
            da_z3_pc.name = "Z3_PC"

            # --- 时间修正（B2000 第二段 run） ---
            if apply_shift and year_int >= 105:
                da_z3_1000 = shift_time_cftime(
                    da_z3_1000.to_dataset(name="Z3_1000"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z3_1000"]

                da_z3_pc = shift_time_cftime(
                    da_z3_pc.to_dataset(name="Z3_PC"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z3_PC"]

            out_1000 = da_z3_1000.compute()
            out_1000.name = "Z3_1000"

            out_pc = da_z3_pc.compute()
            out_pc.name = "Z3_PC"
            out_pc["lev"].attrs["units"] = "hPa"
            out_pc["lev"].attrs["long_name"] = "pressure"

            return (out_1000, out_pc)

    except Exception as e:
        print(f"❌ Error in reference year {year_int:04d}: {str(e)}")
        return None


# ============================================================
# 5. 单文件 hindcast 处理
# ============================================================
def process_one_hindcast_file(z3_file, ps_file):
    if not (os.path.exists(z3_file) and os.path.exists(ps_file)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        with xr.open_dataset(z3_file, decode_times=time_coder) as ds_z, \
             xr.open_dataset(ps_file, decode_times=time_coder) as ds_ps:

            # ------------------------------------------------------------
            # 先构造全层 pressure (Pa)
            # ------------------------------------------------------------
            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]

            p3d = hyam * p0 + hybm * ps

            # 只保留 20N-90N
            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, 90.0))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, 90.0))

            # ------------------------------------------------------------
            # 核心：先插值/外推到固定 pressure levels
            # ------------------------------------------------------------
            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            # --- A. AO: 1000 hPa ---
            da_z3_1000 = z3_plev_nh.sel(lev=AO_PLEV_HPA)
            da_z3_1000.name = "Z3_1000"

            # --- B. NAM: 极冠平均多层高度场 ---
            z3_polar = z3_plev_nh.sel(lat=slice(LAT_MIN_NAM, LAT_MAX_NAM))
            z3_pc_lonmean = z3_polar.mean("lon")
            weights = np.cos(np.deg2rad(z3_pc_lonmean["lat"]))
            da_z3_pc = z3_pc_lonmean.weighted(weights).mean("lat")
            da_z3_pc.name = "Z3_PC"

            out_1000 = da_z3_1000.compute()
            out_1000.name = "Z3_1000"

            out_pc = da_z3_pc.compute()
            out_pc.name = "Z3_PC"
            out_pc["lev"].attrs["units"] = "hPa"
            out_pc["lev"].attrs["long_name"] = "pressure"

            return (out_1000, out_pc)

    except Exception as e:
        print(f"❌ Error in hindcast file:\n   Z3={z3_file}\n   PS={ps_file}\n   {str(e)}")
        return None


# ============================================================
# 6. 构建参考气候态 / EOF（只做一次）
# ============================================================
def build_reference_from_b2000():
    print("\n" + "=" * 80)
    print("🌟 Step 1: 正在构建 B2000WCN001002 参考气候态与 AO EOF 求解器 ...")
    print("=" * 80)

    base_dir = REF_EXP["base_dir"]
    file_template = REF_EXP["file_template"]
    apply_shift = REF_EXP["apply_shift"]

    z3_files = glob.glob(os.path.join(base_dir, "Z3", "*.nc"))
    if not z3_files:
        z3_files = glob.glob(os.path.join(base_dir, "*.Z3.nc"))

    max_year = len(z3_files) if len(z3_files) > 0 else 250
    years_to_process = list(range(1, max_year + 1))
    print(f"--> 参考组预计扫描 {len(years_to_process)} 年")

    args_list = [(y, base_dir, apply_shift, file_template) for y in years_to_process]

    with Pool(processes=CORES) as pool:
        results = pool.map(_process_ref_wrapper, args_list)

    valid_results = [res for res in results if res is not None]
    if not valid_results:
        raise RuntimeError("未能成功构建 B2000WCN001002 参考数据，请检查原始文件。")

    z3_1000_all = xr.concat([res[0] for res in valid_results], dim="time").sortby("time")
    z3_pc_all = xr.concat([res[1] for res in valid_results], dim="time").sortby("time")
    z3_pc_all["lev"].attrs["units"] = "hPa"
    z3_pc_all["lev"].attrs["long_name"] = "pressure"

    del results, valid_results
    gc.collect()

    # ---------------- AO 参考 ----------------
    clim_1000 = z3_1000_all.groupby("time.dayofyear").mean("time")
    clim_1000_smooth = (
        clim_1000.rolling(dayofyear=21, center=True)
        .mean()
        .bfill("dayofyear")
        .ffill("dayofyear")
    )

    daily_anom = z3_1000_all.groupby("time.dayofyear") - clim_1000_smooth
    daily_anom = drop_coord_if_present(daily_anom, "dayofyear")

    monthly_anom = daily_anom.resample(time="1MS").mean().dropna(dim="time", how="all")

    coslat = np.clip(np.cos(np.deg2rad(monthly_anom.lat)), 0, None)
    wgts_2d = np.sqrt(coslat).broadcast_like(
        monthly_anom.isel(time=0).drop_vars("time", errors="ignore")
    )

    solver = Eof(monthly_anom, weights=wgts_2d.values)
    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze()

    if float(eof1_reg.sel(lat=slice(75, 90)).mean().values) > 0:
        flip_factor = -1
        pc1_raw = pc1_raw * -1
    else:
        flip_factor = 1

    sigma_monthly = pc1_raw.std(dim="time")
    explained_var = float(solver.varianceFraction(neigs=1)[0].values * 100)

    # ---------------- NAM 参考 ----------------
    clim_pc = z3_pc_all.groupby("time.dayofyear").mean("time")
    clim_pc_smooth = (
        clim_pc.rolling(dayofyear=21, center=True)
        .mean()
        .bfill("dayofyear")
        .ffill("dayofyear")
    )

    anom_pc = z3_pc_all.groupby("time.dayofyear") - clim_pc_smooth
    anom_pc = drop_coord_if_present(anom_pc, "dayofyear")
    std_pc = anom_pc.std("time")

    REF_DATA["clim_1000_smooth"] = clim_1000_smooth
    REF_DATA["solver"] = solver
    REF_DATA["flip_factor"] = flip_factor
    REF_DATA["sigma_monthly"] = sigma_monthly
    REF_DATA["clim_pc_smooth"] = clim_pc_smooth
    REF_DATA["std_pc"] = std_pc

    print(f"✅ 参考 AO EOF 已构建完成，第一模态解释方差 = {explained_var:.2f}%")
    print("✅ 参考 NAM 气候态 / 标准差 已缓存")
    print("=" * 80)

    del z3_1000_all, z3_pc_all, daily_anom, monthly_anom, anom_pc
    gc.collect()


# ============================================================
# 7. Hindcast 文件配对
# ============================================================
def get_hindcast_experiment_dirs():
    exp_dirs = sorted([
        d for d in os.listdir(HINDCAST_ROOT)
        if os.path.isdir(os.path.join(HINDCAST_ROOT, d))
    ])

    if not INCLUDE_NOCOUPL_HINDCAST:
        exp_dirs = [d for d in exp_dirs if "_NOCOUPL" not in d]

    return exp_dirs


def build_hindcast_file_pairs(exp_dir):
    z3_dir = os.path.join(exp_dir, "Z3")
    ps_dir = os.path.join(exp_dir, "PS")

    if not os.path.isdir(z3_dir):
        print(f"⚠️ 缺少 Z3 目录: {z3_dir}")
        return [], []

    if not os.path.isdir(ps_dir):
        print(f"⚠️ 缺少 PS 目录: {ps_dir}")
        return [], []

    z3_files = sorted(glob.glob(os.path.join(z3_dir, "*.nc")))
    if not z3_files:
        print(f"⚠️ 未找到 Z3 nc 文件: {z3_dir}")
        return [], []

    pairs = []
    missing_ps = []

    for z3_file in z3_files:
        z3_name = os.path.basename(z3_file)
        ps_name = re.sub(r"\.Z3\.nc$", ".PS.nc", z3_name)
        ps_file = os.path.join(ps_dir, ps_name)

        if os.path.exists(ps_file):
            pairs.append((z3_file, ps_file))
        else:
            missing_ps.append((z3_name, ps_name))

    return pairs, missing_ps


# ============================================================
# 8. 处理单个 Hindcast 实验
# ============================================================
def process_one_hindcast_experiment(exp_name):
    exp_path = os.path.join(HINDCAST_ROOT, exp_name)
    out_dir = os.path.join(exp_path, "NAM")
    os.makedirs(out_dir, exist_ok=True)

    print("\n" + "🚀" * 25)
    print(f"正在处理 Hindcast 实验: {exp_name}")
    print("🚀" * 25)

    pairs, missing_ps = build_hindcast_file_pairs(exp_path)

    if missing_ps:
        print(f"⚠️ {exp_name} 有 {len(missing_ps)} 个 Z3 文件未找到对应 PS，将跳过这些文件。")
        for z3_name, expected_ps in missing_ps[:5]:
            print(f"   - Z3: {z3_name}")
            print(f"     预期PS: {expected_ps}")

    if not pairs:
        print(f"❌ {exp_name} 没有可用的 Z3/PS 配对文件，跳过。")
        return

    print(f"--> 找到 {len(pairs)} 对有效 Z3/PS 文件")

    with Pool(processes=CORES) as pool:
        results = pool.map(_process_hindcast_wrapper, pairs)

    valid_results = [res for res in results if res is not None]
    if not valid_results:
        print(f"❌ {exp_name} 所有文件处理失败，跳过。")
        return

    z3_1000_all = xr.concat([res[0] for res in valid_results], dim="time").sortby("time")
    z3_pc_all = xr.concat([res[1] for res in valid_results], dim="time").sortby("time")
    z3_pc_all["lev"].attrs["units"] = "hPa"
    z3_pc_all["lev"].attrs["long_name"] = "pressure"

    del results, valid_results
    gc.collect()

    # ---------------- AO ----------------
    print(f"🌟 开始计算 {exp_name} 的 AO 指数（基于 B2000WCN 参考 EOF）...")

    daily_anom = z3_1000_all.groupby("time.dayofyear") - REF_DATA["clim_1000_smooth"]
    daily_anom = drop_coord_if_present(daily_anom, "dayofyear")

    pseudo_pcs_raw = (
        REF_DATA["solver"].projectField(daily_anom, neofs=1, eofscaling=0).squeeze()
        * REF_DATA["flip_factor"]
    )
    ao_index_daily = pseudo_pcs_raw / REF_DATA["sigma_monthly"]
    ao_index_daily.name = "AO_Index"
    ao_index_daily.attrs["long_name"] = f"Daily AO index for hindcast {exp_name}"
    ao_index_daily.attrs["reference_climatology"] = "B2000WCN001002"

    ao_csv_path = os.path.join(out_dir, f"{exp_name}_Daily_AO_Index.csv")
    ao_nc_path = os.path.join(out_dir, f"{exp_name}_Daily_AO_Index.nc")

    pd.DataFrame({
        "Date": ao_index_daily.time.values,
        "AO_Index": ao_index_daily.values
    }).to_csv(ao_csv_path, index=False)

    ao_index_daily.to_netcdf(ao_nc_path)

    print(f"✅ AO CSV 已保存: {ao_csv_path}")
    print(f"✅ AO NC  已保存: {ao_nc_path}")

    # ---------------- NAM ----------------
    print(f"🌟 开始计算 {exp_name} 的垂直 NAM 指数（基于 B2000WCN 参考气候态）...")

    anom_pc = z3_pc_all.groupby("time.dayofyear") - REF_DATA["clim_pc_smooth"]
    anom_pc = drop_coord_if_present(anom_pc, "dayofyear")

    nam_vertical = -(anom_pc / REF_DATA["std_pc"])
    nam_vertical.name = "NAM_Vertical"
    nam_vertical.attrs["long_name"] = f"Vertical NAM index for hindcast {exp_name}"
    nam_vertical.attrs["reference_climatology"] = "B2000WCN001002"
    nam_vertical["lev"].attrs["units"] = "hPa"
    nam_vertical["lev"].attrs["long_name"] = "pressure"

    nam_nc_path = os.path.join(out_dir, f"{exp_name}_Vertical_NAM.nc")
    nam_vertical.to_netcdf(nam_nc_path)

    print(f"✅ NAM 已保存: {nam_nc_path}")

    del z3_1000_all, z3_pc_all, daily_anom, anom_pc, ao_index_daily, nam_vertical
    gc.collect()


# ============================================================
# 9. 主程序
# ============================================================
if __name__ == "__main__":
    build_reference_from_b2000()

    exp_dirs = get_hindcast_experiment_dirs()

    print("\n" + "=" * 80)
    print("🌟 Step 2: 开始批量处理 Hindcast AO / NAM")
    print("=" * 80)
    print(f"--> 共找到 {len(exp_dirs)} 个待处理实验")
    for d in exp_dirs:
        print(f"   {d}")

    for exp_name in exp_dirs:
        process_one_hindcast_experiment(exp_name)

    print("\n🎉 所有 Hindcast 实验 AO / NAM 处理完成！")

In [ ]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from cartopy.util import add_cyclic_point
from eofs.xarray import Eof

# ================= 配置区 =================
WACCM6_FILE = "/mnt/soclim0/andreas/zg_Amon_CESM2-WACCM_amip_r1i1p1f1_gn_195001-201412.nc"
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# NOAA 风格配色与边界
noaa_colors = [
    "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5", "#a3dcfb", "#c2f0fb",
    "#dff8fd", "#f0fcfd", "#ffffff", "#fefbd9", "#feea9e", "#fec762",
    "#f27932", "#b82522"
]
bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
cmap_noaa = mcolors.ListedColormap(noaa_colors)
norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)


# ================= 核心外推函数 =================
def _extrapolate_to_1000hpa_1d(z_prof, p_prof):
    """
    针对单个格点列的 1D 外推函数
    """
    target_p = 100000.0  # 目标 1000 hPa (即 100000 Pa)
    
    # 过滤掉 NaN 的层 (比如地形以下的层)
    mask = np.isfinite(z_prof) & np.isfinite(p_prof) & (p_prof > 0)
    if not np.any(mask):
        return np.nan
        
    z = z_prof[mask]
    p = p_prof[mask]
    
    if len(z) < 2:
        return z[0] if len(z) == 1 else np.nan
        
    # 按气压从小到大排序 (高空到低空)
    order = np.argsort(p)
    p = p[order]
    z = z[order]
    
    lp = np.log(p)
    ltp = np.log(target_p)
    
    # 向下外推 (更底层，气压更大，比如青藏高原向下推到 1000hPa)
    if ltp > lp[-1]:
        slope = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        return z[-1] + slope * (ltp - lp[-1])
    # 向上外推 (极少出现，防错)
    elif ltp < lp[0]:
        slope = (z[1] - z[0]) / (lp[1] - lp[0])
        return z[0] + slope * (ltp - lp[0])
    # 如果目标层就在数据范围内，直接内插
    else:
        return np.interp(ltp, lp, z)

def extrapolate_zg_to_1000(zg_da):
    """
    利用 xarray 的 apply_ufunc 将 1D 外推应用到整个多维数组
    """
    # 提取坐标
    plev_da = zg_da.plev
    
    print("   --> 正在执行底层外推运算 (使用 Numpy 矢量化)...")
    zg_1000_da = xr.apply_ufunc(
        _extrapolate_to_1000hpa_1d,
        zg_da,
        plev_da,
        input_core_dims=[['plev'], ['plev']],
        output_core_dims=[[]],
        vectorize=True,
        output_dtypes=[np.float64]
    )
    
    zg_1000_da.name = "ZG_1000"
    return zg_1000_da

# ================= EOF 计算 =================
def compute_eof_regression(da_monthly):
    # 计算月度距平
    clim = da_monthly.groupby("time.month").mean("time")
    anom = (da_monthly.groupby("time.month") - clim).drop_vars("month")
    
    # 面积加权
    coslat = np.clip(np.cos(np.deg2rad(anom.lat)), 0, None)
    wgts_2d = np.sqrt(coslat).broadcast_like(anom.isel(time=0).drop_vars("time", errors="ignore"))
    
    # EOF 分解
    solver = Eof(anom, weights=wgts_2d.values)
    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    explained_var = solver.varianceFraction(neigs=1)[0].values * 100
    
    # 极区统一为负值，确保模态符号一致
    if eof1_reg.sel(lat=slice(75, 90)).mean() > 0:
        eof1_reg = eof1_reg * -1
        
    return eof1_reg, explained_var

# ================= 主程序 =================
if __name__ == "__main__":
    print(f"🔄 正在读取 WACCM6 数据: {WACCM6_FILE}")
    
    ds_w6 = xr.open_dataset(WACCM6_FILE, use_cftime=True)
    
    # 【第一步】：提取全层数据到内存中，仅保留北半球以加速
    print("⬇️  正在将全垂直层的北半球数据装载入内存...")
    zg_all_lev = ds_w6["zg"].sel(lat=slice(20.0, 90.0)).compute()
    
    # 【诊断】：打印外推前的 NaN 数量 (仅统计 1000 hPa 这一层)
    raw_1000 = zg_all_lev.sel(plev=100000)
    nan_before = np.isnan(raw_1000).sum().values
    print(f"📉 外推前，1000hPa 层的 NaN (缺测) 数量为: {nan_before}")
    
    # 【第二步】：执行外推修复
    print("🛠️  开始进行垂直向下外推...")
    da_1000_fixed = extrapolate_zg_to_1000(zg_all_lev)
    
    nan_after = np.isnan(da_1000_fixed).sum().values
    print(f"📈 外推后，1000hPa 层的 NaN (缺测) 数量为: {nan_after}")
    
    if nan_after > 0:
        print("⚠️ 警告：仍有残余 NaN 值，可能是因为某些极高海拔地区上空数据极少。")
    else:
        print("✅ 陆地地形数据已全部填补完毕！")

    # 【第三步】：计算 EOF
    print("\n🧮 正在计算 WACCM6 全年 AO...")
    reg_ann, var_ann = compute_eof_regression(da_1000_fixed)

    print("🧮 正在计算 WACCM6 冬季 (DJF) AO...")
    da_1000_djf = da_1000_fixed.sel(time=da_1000_fixed['time.season'] == 'DJF')
    reg_djf, var_djf = compute_eof_regression(da_1000_djf)

    # ================= 绘图 (1x2 面板) =================
    print("\n🎨 正在绘制修复后的测试图...")
    fig, axes = plt.subplots(
        nrows=1, ncols=2, figsize=(16, 8),
        subplot_kw={'projection': ccrs.NorthPolarStereo()}
    )

    plot_data = [
        (reg_ann, var_ann, "WACCM6 Annual AO (Extrapolated)", axes[0]),
        (reg_djf, var_djf, "WACCM6 DJF AO (Extrapolated)", axes[1])
    ]

    for eof_reg, var, title, ax in plot_data:
        ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)
        
        eof_cyclic, lon_cyclic = add_cyclic_point(eof_reg.values, coord=eof_reg.lon)
        
        cf = ax.contourf(
            lon_cyclic, eof_reg.lat, eof_cyclic,
            levels=bounds,
            transform=ccrs.PlateCarree(),
            cmap=cmap_noaa,
            norm=norm_noaa,
            extend="both"
        )
        
        ax.set_title(f"{title}\nVar: {var:.1f}%", fontsize=16, fontweight="bold", pad=15)

    cbar_ax = fig.add_axes([0.15, 0.1, 0.7, 0.04])
    cbar = fig.colorbar(
        cf, cax=cbar_ax, orientation="horizontal",
        ticks=[-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
    )
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label("1000hPa height regression (m)", fontsize=14)

    plt.subplots_adjust(wspace=0.1, bottom=0.2)

    fig_path = os.path.join(FIG_OUT_DIR, "WACCM6_AO_Debug_Fixed.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"✅ 调试绘图完成！修复后的图片已保存至: {fig_path}")

    plt.show()

In [ ]:
import os
import gc
import glob
import xarray as xr
import numpy as np
import pandas as pd
from multiprocessing import Pool
from eofs.xarray import Eof
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from cartopy.util import add_cyclic_point

# ================= 1. 核心配置区 =================
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# 数据路径
WACCM4_BASE_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002"
WACCM6_FILE = "/mnt/soclim0/andreas/zg_Amon_CESM2-WACCM_amip_r1i1p1f1_gn_195001-201412.nc"

LAT_MIN_AO = 20.0
CORES = 10
SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 104

# 【极大加速】：因为只算 AO，所以只插值 1000 hPa 这一层
TARGET_PLEV_HPA = np.array([1000.0], dtype=np.float64)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

# NOAA 风格配色与边界
noaa_colors = [
    "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5", "#a3dcfb", "#c2f0fb",
    "#dff8fd", "#f0fcfd", "#ffffff", "#fefbd9", "#feea9e", "#fec762",
    "#f27932", "#b82522"
]
bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
cmap_noaa = mcolors.ListedColormap(noaa_colors)
norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)


# ================= 2. 基础工具与插值函数 =================
def shift_time_cftime(ds, year_offset):
    times = ds["time"].values
    new_times = [t.replace(year=t.year + year_offset) for t in times]
    ds["time"] = xr.DataArray(new_times, coords=[new_times], dims=["time"])
    return ds

def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)

    out = np.interp(ltp, lp, z)

    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out

def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )

    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    return z_on_plev

# --- 新增: 专为 WACCM6 设计的外推函数 ---
def _extrapolate_to_1000hpa_1d(z_prof, p_prof):
    target_p = 100000.0
    mask = np.isfinite(z_prof) & np.isfinite(p_prof) & (p_prof > 0)
    if not np.any(mask):
        return np.nan
        
    z = z_prof[mask]
    p = p_prof[mask]
    
    if len(z) < 2:
        return z[0] if len(z) == 1 else np.nan
        
    order = np.argsort(p)
    p = p[order]
    z = z[order]
    
    lp = np.log(p)
    ltp = np.log(target_p)
    
    if ltp > lp[-1]:
        slope = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        return z[-1] + slope * (ltp - lp[-1])
    elif ltp < lp[0]:
        slope = (z[1] - z[0]) / (lp[1] - lp[0])
        return z[0] + slope * (ltp - lp[0])
    else:
        return np.interp(ltp, lp, z)

def extrapolate_zg_to_1000(zg_da):
    plev_da = zg_da.plev
    zg_1000_da = xr.apply_ufunc(
        _extrapolate_to_1000hpa_1d,
        zg_da,
        plev_da,
        input_core_dims=[['plev'], ['plev']],
        output_core_dims=[[]],
        vectorize=True,
        output_dtypes=[np.float64]
    )
    zg_1000_da.name = "ZG_1000"
    return zg_1000_da
# ----------------------------------------

def compute_eof_regression(da_monthly):
    """仅计算 EOF1 回归场及方差贡献"""
    clim = da_monthly.groupby("time.month").mean("time")
    anom = (da_monthly.groupby("time.month") - clim).drop_vars("month")
    
    coslat = np.clip(np.cos(np.deg2rad(anom.lat)), 0, None)
    wgts_2d = np.sqrt(coslat).broadcast_like(anom.isel(time=0).drop_vars("time", errors="ignore"))
    
    solver = Eof(anom, weights=wgts_2d.values)
    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    explained_var = solver.varianceFraction(neigs=1)[0].values * 100
    
    if eof1_reg.sel(lat=slice(75, 90)).mean() > 0:
        eof1_reg = eof1_reg * -1
        
    return eof1_reg, explained_var

# ================= 3. WACCM4 单年处理函数 =================
def process_waccm4_year(year_int):
    f_z3_name = f"B2000WCN.sample.cam.h3.{year_int:04d}.Z3.nc"
    f_ps_name = f"B2000WCN.sample.cam.h3.{year_int:04d}.PS.nc"

    f_z3 = os.path.join(WACCM4_BASE_DIR, "Z3", f_z3_name)
    if not os.path.exists(f_z3):
        f_z3 = os.path.join(WACCM4_BASE_DIR, f_z3_name)

    f_ps = os.path.join(WACCM4_BASE_DIR, "PS", f_ps_name)
    if not os.path.exists(f_ps):
        f_ps = os.path.join(WACCM4_BASE_DIR, f_ps_name)

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]
            p3d = hyam * p0 + hybm * ps

            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, 90.0))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, 90.0))

            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh, p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA, target_p_hpa=TARGET_PLEV_HPA
            )

            da_z3_1000 = z3_plev_nh.sel(lev=1000.0)
            da_z3_1000.name = "Z3_1000"

            if year_int >= 105:
                da_z3_1000 = shift_time_cftime(
                    da_z3_1000.to_dataset(name="Z3_1000"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z3_1000"]

            return da_z3_1000.compute()

    except Exception as e:
        print(f"Error in WACCM4 year {year_int:04d}: {str(e)}")
        return None

# ================= 4. 主程序 (独立提取与对比绘制) =================
if __name__ == "__main__":
    print("🚀 开始读取并处理 WACCM4 (B2000WCN) 数据...")
    # 扫描 WACCM4 年数
    z3_files = glob.glob(os.path.join(WACCM4_BASE_DIR, "Z3", "*.nc"))
    if not z3_files:
        z3_files = glob.glob(os.path.join(WACCM4_BASE_DIR, "*.Z3.nc"))
    max_year = len(z3_files) if len(z3_files) > 0 else 250
    years_to_process = list(range(1, max_year + 1))

    # 多进程提取 1000 hPa
    with Pool(processes=CORES) as pool:
        results = pool.map(process_waccm4_year, years_to_process)
        
    valid_results = [res for res in results if res is not None]
    z3_1000_daily_all = xr.concat(valid_results, dim="time").sortby("time")
    del results, valid_results
    gc.collect()

    print("🔄 正在将 WACCM4 日均数据重采样为月均...")
    z3_1000_monthly = z3_1000_daily_all.resample(time="1MS").mean().dropna(dim="time", how="all")

    print("🔄 正在读取 WACCM6 Amon 数据并装载入内存...")
    ds_w6 = xr.open_dataset(WACCM6_FILE, use_cftime=True)
    
    # ------------------ 修改的 WACCM6 外推逻辑 ------------------
    print("⬇️  正在将 WACCM6 全垂直层数据装载入内存...")
    zg_all_lev = ds_w6["zg"].sel(lat=slice(20.0, 90.0)).compute()
    
    print("🛠️  开始进行 WACCM6 垂直向下外推以填补地形缺测...")
    waccm6_1000_monthly = extrapolate_zg_to_1000(zg_all_lev)
    # -----------------------------------------------------------

    print("🧮 正在计算 WACCM4 的 EOF (全年与冬季)...")
    w4_ann_reg, w4_ann_var = compute_eof_regression(z3_1000_monthly)
    z3_1000_djf = z3_1000_monthly.sel(time=z3_1000_monthly['time.season'] == 'DJF')
    w4_djf_reg, w4_djf_var = compute_eof_regression(z3_1000_djf)

    print("🧮 正在计算 WACCM6 的 EOF (全年与冬季)...")
    w6_ann_reg, w6_ann_var = compute_eof_regression(waccm6_1000_monthly)
    waccm6_1000_djf = waccm6_1000_monthly.sel(time=waccm6_1000_monthly['time.season'] == 'DJF')
    w6_djf_reg, w6_djf_var = compute_eof_regression(waccm6_1000_djf)

    # ================= 绘图 (2x2 面板) =================
    print("🎨 正在绘制四图对比面板...")
    fig, axes = plt.subplots(
        nrows=2, ncols=2, figsize=(16, 16),
        subplot_kw={'projection': ccrs.NorthPolarStereo()}
    )

    plot_data = [
        (w4_ann_reg, w4_ann_var, "WACCM4 Annual AO", axes[0, 0]),
        (w6_ann_reg, w6_ann_var, "WACCM6 Annual AO", axes[0, 1]),
        (w4_djf_reg, w4_djf_var, "WACCM4 DJF AO", axes[1, 0]),
        (w6_djf_reg, w6_djf_var, "WACCM6 DJF AO", axes[1, 1])
    ]

    for eof_reg, var, title, ax in plot_data:
        ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)
        
        eof_cyclic, lon_cyclic = add_cyclic_point(eof_reg.values, coord=eof_reg.lon)
        
        cf = ax.contourf(
            lon_cyclic, eof_reg.lat, eof_cyclic,
            levels=bounds,
            transform=ccrs.PlateCarree(),
            cmap=cmap_noaa,
            norm=norm_noaa,
            extend="both"
        )
        
        ax.set_title(f"{title}\nVar: {var:.1f}%", fontsize=16, fontweight="bold", pad=15)

    # 添加全局共享 Colorbar
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
    cbar = fig.colorbar(
        cf, cax=cbar_ax, orientation="horizontal",
        ticks=[-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
    )
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label("1000mb height regression (m)", fontsize=14)

    plt.subplots_adjust(wspace=0.1, hspace=0.2, bottom=0.12)

    fig_path = os.path.join(FIG_OUT_DIR, "WACCM4_vs_WACCM6_AO_Comparison.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"✅ 绘图完成！图片已保存至: {fig_path}")

    plt.show()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ======================================================================
# Standalone script:
# 1) Rebuild / load Z1000 monthly fields for WACCM4 / WACCM6 / ERA5
# 2) Compute AO EOF regression maps
#    - WACCM4 / WACCM6: keep original monthly EOF-regression logic
#    - ERA5: strictly follow the validation-pipeline regression-map logic
#            (1979-2000 base-period monthly EOF, pcscaling=1)
# 3) Force the 75-90N polar-cap mean of every regression map to be NEGATIVE
# 4) Plot 2x3 comparison figure:
#    [WACCM4 Annual, WACCM6 Annual, ERA5 Annual]
#    [WACCM4 DJFM , WACCM6 DJFM , ERA5 DJFM ]
# ======================================================================

import os
import gc
import re
import glob
import dask
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from multiprocessing import Pool
from dask.diagnostics import ProgressBar
from eofs.xarray import Eof

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point


# ======================================================================
# 1. Config
# ======================================================================

dask.config.set(scheduler="threads", num_workers=8)

FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

CACHE_DIR = os.path.join(FIG_OUT_DIR, "z1000_monthly_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# -----------------------------
# Raw data paths
# -----------------------------
WACCM4_BASE_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002"
WACCM6_FILE = "/mnt/soclim0/andreas/zg_Amon_CESM2-WACCM_amip_r1i1p1f1_gn_195001-201412.nc"
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"

# -----------------------------
# Outputs
# -----------------------------
OUT_FIG = os.path.join(FIG_OUT_DIR, "WACCM4_vs_WACCM6_vs_ERA5_AO_Comparison_DJFM.png")

WACCM4_CACHE = os.path.join(CACHE_DIR, "WACCM4_Z1000_monthly.nc")
WACCM6_CACHE = os.path.join(CACHE_DIR, "WACCM6_Z1000_monthly.nc")
ERA5_CACHE   = os.path.join(CACHE_DIR, "ERA5_Z1000_monthly.nc")

WACCM4_ANN_EOF_NC  = os.path.join(CACHE_DIR, "WACCM4_Annual_EOF1_Regression.nc")
WACCM4_DJFM_EOF_NC = os.path.join(CACHE_DIR, "WACCM4_DJFM_EOF1_Regression.nc")
WACCM6_ANN_EOF_NC  = os.path.join(CACHE_DIR, "WACCM6_Annual_EOF1_Regression.nc")
WACCM6_DJFM_EOF_NC = os.path.join(CACHE_DIR, "WACCM6_DJFM_EOF1_Regression.nc")
ERA5_ANN_EOF_NC    = os.path.join(CACHE_DIR, "ERA5_Annual_EOF1_Regression_STRICT.nc")
ERA5_DJFM_EOF_NC   = os.path.join(CACHE_DIR, "ERA5_DJFM_EOF1_Regression_STRICT.nc")

# -----------------------------
# Domain / years
# -----------------------------
LAT_MIN_AO = 20.0
LAT_MAX_AO = 90.0

ERA5_YEAR_MIN = 1950
ERA5_YEAR_MAX = 2020
ERA5_BASE_START = "1979-01-01"
ERA5_BASE_END   = "2000-12-31"

CORES = 10
SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 104
G = 9.80665

# WACCM4 interpolation target
TARGET_PLEV_HPA = np.array([1000.0], dtype=np.float64)
TARGET_PLEV_PA  = TARGET_PLEV_HPA * 100.0

# NOAA style colorbar
noaa_colors = [
    "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5", "#a3dcfb", "#c2f0fb",
    "#dff8fd", "#f0fcfd", "#ffffff", "#fefbd9", "#feea9e", "#fec762",
    "#f27932", "#b82522"
]
bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
cmap_noaa = mcolors.ListedColormap(noaa_colors)
norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)


# ======================================================================
# 2. Utility functions
# ======================================================================

def shift_time_cftime(ds, year_offset):
    times = ds["time"].values
    new_times = [t.replace(year=t.year + year_offset) for t in times]
    ds = ds.copy()
    ds["time"] = xr.DataArray(new_times, coords=[new_times], dims=["time"])
    return ds


def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)

    out = np.interp(ltp, lp, z)

    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out


def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )

    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    z_on_plev["lev"].attrs["units"] = "hPa"
    z_on_plev["lev"].attrs["long_name"] = "pressure"
    return z_on_plev


def _extrapolate_to_1000hpa_1d(z_prof, p_prof):
    target_p = 100000.0

    z_prof = np.asarray(z_prof, dtype=np.float64)
    p_prof = np.asarray(p_prof, dtype=np.float64)

    mask = np.isfinite(z_prof) & np.isfinite(p_prof) & (p_prof > 0)
    if not np.any(mask):
        return np.nan

    z = z_prof[mask]
    p = p_prof[mask]

    if len(z) < 2:
        return z[0] if len(z) == 1 else np.nan

    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(target_p)

    if ltp > lp[-1]:
        slope = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        return z[-1] + slope * (ltp - lp[-1])
    elif ltp < lp[0]:
        slope = (z[1] - z[0]) / (lp[1] - lp[0])
        return z[0] + slope * (ltp - lp[0])
    else:
        return np.interp(ltp, lp, z)


def extrapolate_zg_to_1000(zg_da):
    plev_da = zg_da["plev"]
    zg_1000_da = xr.apply_ufunc(
        _extrapolate_to_1000hpa_1d,
        zg_da,
        plev_da,
        input_core_dims=[["plev"], ["plev"]],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[np.float64]
    )
    zg_1000_da.name = "Z1000"
    return zg_1000_da


def select_djfm(da_monthly):
    return da_monthly.sel(time=da_monthly["time.month"].isin([12, 1, 2, 3]))


def polar_cap_weighted_mean(da, lat_name="lat", lon_name="lon", lat_min=75.0, lat_max=90.0):
    """
    Area-weighted polar-cap mean over 75-90N.
    Robust to latitude ascending/descending order.
    For a 2D field (lat, lon).
    """
    lat = da[lat_name]
    lat_mask = (lat >= lat_min) & (lat <= lat_max)
    da_cap = da.where(lat_mask, drop=True)

    if da_cap.sizes.get(lat_name, 0) == 0:
        raise ValueError(f"No latitude points found in {lat_min}-{lat_max}N.")

    weights = np.cos(np.deg2rad(da_cap[lat_name]))
    return da_cap.weighted(weights).mean(dim=(lat_name, lon_name))


def enforce_negative_polar_cap(eof_reg, lat_name="lat", lon_name="lon", lat_min=75.0, lat_max=90.0):
    """
    Force the 75-90N polar-cap mean of eof_reg to be negative.
    """
    pc_mean = float(
        polar_cap_weighted_mean(
            eof_reg,
            lat_name=lat_name,
            lon_name=lon_name,
            lat_min=lat_min,
            lat_max=lat_max
        ).values
    )

    if pc_mean > 0:
        eof_reg = eof_reg * -1.0

    return eof_reg


def plot_one_panel(ax, eof_reg, var_pct, title):
    ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
    ax.coastlines(linewidth=1)
    ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

    eof_cyclic, lon_cyclic = add_cyclic_point(eof_reg.values, coord=eof_reg["lon"])

    cf = ax.contourf(
        lon_cyclic, eof_reg["lat"], eof_cyclic,
        levels=bounds,
        transform=ccrs.PlateCarree(),
        cmap=cmap_noaa,
        norm=norm_noaa,
        extend="both"
    )

    ax.set_title(
        f"{title}\nVar: {var_pct:.1f}%",
        fontsize=15,
        fontweight="bold",
        pad=14
    )
    return cf


# ======================================================================
# 3. WACCM4 monthly Z1000
# ======================================================================

def process_waccm4_year(year_int):
    f_z3_name = f"B2000WCN.sample.cam.h3.{year_int:04d}.Z3.nc"
    f_ps_name = f"B2000WCN.sample.cam.h3.{year_int:04d}.PS.nc"

    f_z3 = os.path.join(WACCM4_BASE_DIR, "Z3", f_z3_name)
    if not os.path.exists(f_z3):
        f_z3 = os.path.join(WACCM4_BASE_DIR, f_z3_name)

    f_ps = os.path.join(WACCM4_BASE_DIR, "PS", f_ps_name)
    if not os.path.exists(f_ps):
        f_ps = os.path.join(WACCM4_BASE_DIR, f_ps_name)

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]
            p3d = hyam * p0 + hybm * ps

            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, LAT_MAX_AO))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, LAT_MAX_AO))

            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            da_z3_1000 = z3_plev_nh.sel(lev=1000.0)
            da_z3_1000.name = "Z1000"

            if year_int >= 105:
                da_z3_1000 = shift_time_cftime(
                    da_z3_1000.to_dataset(name="Z1000"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z1000"]

            return da_z3_1000.compute()

    except Exception as e:
        print(f"Error in WACCM4 year {year_int:04d}: {e}")
        return None


def build_waccm4_monthly(cache_path=WACCM4_CACHE, overwrite=False):
    if os.path.exists(cache_path) and not overwrite:
        print(f"📦 Loading cached WACCM4 monthly Z1000: {cache_path}")
        ds = xr.open_dataset(cache_path, use_cftime=True)
        return ds["Z1000"]

    print("\n🚀 Building WACCM4 monthly Z1000 from raw files...")

    z3_files = glob.glob(os.path.join(WACCM4_BASE_DIR, "Z3", "*.nc"))
    if not z3_files:
        z3_files = glob.glob(os.path.join(WACCM4_BASE_DIR, "*.Z3.nc"))

    max_year = len(z3_files) if len(z3_files) > 0 else 250
    years_to_process = list(range(1, max_year + 1))
    print(f"--> WACCM4 candidate years: {len(years_to_process)}")

    with Pool(processes=CORES) as pool:
        results = pool.map(process_waccm4_year, years_to_process)

    valid_results = [res for res in results if res is not None]
    if len(valid_results) == 0:
        raise RuntimeError("No valid WACCM4 Z1000 data found.")

    z1000_daily_all = xr.concat(valid_results, dim="time").sortby("time")
    z1000_daily_all.name = "Z1000"

    print("🔄 Resampling WACCM4 daily -> monthly ...")
    z1000_monthly = z1000_daily_all.resample(time="1MS").mean().dropna(dim="time", how="all")
    z1000_monthly.name = "Z1000"
    z1000_monthly.attrs["long_name"] = "WACCM4 1000 hPa geopotential height monthly mean"
    z1000_monthly.attrs["units"] = getattr(z1000_daily_all, "units", "m")

    z1000_monthly.to_dataset(name="Z1000").to_netcdf(cache_path)
    print(f"✅ Saved WACCM4 monthly cache: {cache_path}")

    del results, valid_results, z1000_daily_all
    gc.collect()

    return z1000_monthly


# ======================================================================
# 4. WACCM6 monthly Z1000
# ======================================================================

def build_waccm6_monthly(cache_path=WACCM6_CACHE, overwrite=False):
    if os.path.exists(cache_path) and not overwrite:
        print(f"📦 Loading cached WACCM6 monthly Z1000: {cache_path}")
        ds = xr.open_dataset(cache_path, use_cftime=True)
        return ds["Z1000"]

    print("\n🚀 Building WACCM6 monthly Z1000 from raw file...")

    ds_w6 = xr.open_dataset(WACCM6_FILE, use_cftime=True)
    zg = ds_w6["zg"].sel(lat=slice(LAT_MIN_AO, LAT_MAX_AO))

    plev_vals = zg["plev"].values
    if np.nanmax(plev_vals) < 2000:
        zg = zg.assign_coords(plev=zg["plev"] * 100.0)
        zg["plev"].attrs["units"] = "Pa"
        print("🔧 WACCM6 plev detected in hPa, converted to Pa.")
    else:
        print("🔧 WACCM6 plev detected in Pa.")

    print("⬇️ Computing WACCM6 Z1000 by downward extrapolation ...")
    with ProgressBar():
        zg_all = zg.compute()
        z1000_monthly = extrapolate_zg_to_1000(zg_all).compute()

    z1000_monthly.name = "Z1000"
    z1000_monthly.attrs["long_name"] = "WACCM6 1000 hPa geopotential height monthly mean"
    z1000_monthly.attrs["units"] = getattr(zg, "units", "m")

    z1000_monthly.to_dataset(name="Z1000").to_netcdf(cache_path)
    print(f"✅ Saved WACCM6 monthly cache: {cache_path}")

    del ds_w6, zg, zg_all
    gc.collect()

    return z1000_monthly


# ======================================================================
# 5. ERA5 monthly Z1000 (built exactly from your pipeline logic)
# ======================================================================

def build_era5_monthly(cache_path=ERA5_CACHE, overwrite=False):
    if os.path.exists(cache_path) and not overwrite:
        print(f"📦 Loading cached ERA5 monthly Z1000: {cache_path}")
        ds = xr.open_dataset(cache_path)
        return ds["Z1000"]

    print("\n🚀 Building ERA5 monthly Z1000 from raw files...")

    pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
    all_files = sorted(glob.glob(os.path.join(ERA5_Z_DIR, "z_4daily*.nc")))

    z_files = [
        f for f in all_files
        if pattern.match(os.path.basename(f))
        and ERA5_YEAR_MIN <= int(pattern.match(os.path.basename(f)).group(1)[:4]) <= ERA5_YEAR_MAX
    ]

    if len(z_files) == 0:
        raise FileNotFoundError("No ERA5 z_4daily files matched.")

    print(f"Matched ERA5 files: {len(z_files)}")

    ds = xr.open_mfdataset(z_files, combine="by_coords", parallel=True, chunks={"time": 180})

    print("⏳ 正在截取 ERA5 1000 hPa 区域并加载到内存...")
    with ProgressBar():
        Z_raw = ds["z"].sel(level=1000, latitude=slice(LAT_MAX_AO, LAT_MIN_AO)).compute()

    mean_val = Z_raw.mean().values
    if mean_val > 500:
        print(f"🔧 检测到 ERA5 数据均值为 {mean_val:.1f}，判定为位势，正在除以 G...")
        Z_1000 = Z_raw / G
    else:
        print(f"🔧 检测到 ERA5 数据均值为 {mean_val:.1f}，判定已经是位势高度，跳过除以 G。")
        Z_1000 = Z_raw

    print("🔄 ERA5 6-hourly -> daily -> monthly ...")
    Z_daily = Z_1000.resample(time="1D").mean()
    Z_monthly = Z_daily.resample(time="1MS").mean()

    Z_monthly = Z_monthly.rename({"latitude": "lat", "longitude": "lon"})
    Z_monthly.name = "Z1000"
    Z_monthly.attrs["long_name"] = "ERA5 1000 hPa geopotential height monthly mean"
    Z_monthly.attrs["units"] = "m"

    Z_monthly.to_dataset(name="Z1000").to_netcdf(cache_path)
    print(f"✅ Saved ERA5 monthly cache: {cache_path}")

    del ds, Z_raw, Z_1000, Z_daily
    gc.collect()

    return Z_monthly


# ======================================================================
# 6. EOF regression functions
# ======================================================================

def compute_model_monthly_regression(monthly_da, season="annual"):
    """
    For WACCM4 / WACCM6:
    monthly EOF regression on the selected monthly sample itself.
    """
    if season.upper() == "ANNUAL":
        work = monthly_da
    elif season.upper() == "DJFM":
        work = select_djfm(monthly_da)
    else:
        raise ValueError("season must be 'annual' or 'DJFM'")

    clim = work.groupby("time.month").mean("time")
    anom = (work.groupby("time.month") - clim).drop_vars("month")

    coslat = np.cos(np.deg2rad(anom["lat"]))
    wgts_2d = np.sqrt(coslat).broadcast_like(anom.isel(time=0))

    solver = Eof(anom, weights=wgts_2d.values)
    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    explained_var = float(solver.varianceFraction(neigs=1)[0].values * 100.0)

    # 强制 75-90N 极圈平均为负
    eof1_reg = enforce_negative_polar_cap(eof1_reg, lat_name="lat", lon_name="lon")

    eof1_reg.name = "EOF1_Regression"
    eof1_reg.attrs["explained_variance_percent"] = explained_var
    return eof1_reg, explained_var


def compute_era5_regression_strict(monthly_da, season="annual"):
    """
    STRICTLY mimic your ERA5 validation-pipeline regression-map logic:
    - use 1979-2000 base period
    - monthly EOF only
    - eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1)
    - then force 75-90N polar-cap mean negative
    """
    base = monthly_da.sel(time=slice(ERA5_BASE_START, ERA5_BASE_END))

    if season.upper() == "ANNUAL":
        base_work = base
    elif season.upper() == "DJFM":
        base_work = base.sel(time=base["time.month"].isin([12, 1, 2, 3]))
    else:
        raise ValueError("season must be 'annual' or 'DJFM'")

    base_work = base_work.dropna(dim="time", how="all")

    clim_monthly = base_work.groupby("time.month").mean("time")
    anom_monthly_base = (base_work.groupby("time.month") - clim_monthly).drop_vars("month")
    anom_monthly_base = anom_monthly_base.dropna(dim="time", how="all")

    if anom_monthly_base.sizes.get("time", 0) == 0:
        raise RuntimeError("ERA5 anomaly has zero valid time samples.")

    coslat = np.cos(np.deg2rad(anom_monthly_base["lat"]))
    wgts_2d = np.sqrt(coslat).broadcast_like(anom_monthly_base.isel(time=0))

    solver = Eof(anom_monthly_base, weights=wgts_2d.values)

    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    explained_var = float(solver.varianceFraction(neigs=1)[0].values * 100.0)

    # 强制 75-90N 极圈平均为负
    eof1_reg = enforce_negative_polar_cap(eof1_reg, lat_name="lat", lon_name="lon")

    eof1_reg.name = "EOF1_Regression"
    eof1_reg.attrs["explained_variance_percent"] = explained_var
    return eof1_reg, explained_var


# ======================================================================
# 7. Main
# ======================================================================

if __name__ == "__main__":

    # --------------------------------------------------
    # Step 1. Build/load monthly Z1000
    # --------------------------------------------------
    w4_monthly = build_waccm4_monthly(cache_path=WACCM4_CACHE, overwrite=False)
    w6_monthly = build_waccm6_monthly(cache_path=WACCM6_CACHE, overwrite=False)
    e5_monthly = build_era5_monthly(cache_path=ERA5_CACHE, overwrite=False)

    # --------------------------------------------------
    # Step 2. Compute regression maps
    # --------------------------------------------------
    print("\n🧮 Computing WACCM4 annual / DJFM regression ...")
    w4_ann_reg,  w4_ann_var  = compute_model_monthly_regression(w4_monthly, season="annual")
    w4_djfm_reg, w4_djfm_var = compute_model_monthly_regression(w4_monthly, season="DJFM")

    print("\n🧮 Computing WACCM6 annual / DJFM regression ...")
    w6_ann_reg,  w6_ann_var  = compute_model_monthly_regression(w6_monthly, season="annual")
    w6_djfm_reg, w6_djfm_var = compute_model_monthly_regression(w6_monthly, season="DJFM")

    print("\n🧮 Computing ERA5 annual / DJFM regression (STRICT validation-pipeline logic) ...")
    e5_ann_reg,  e5_ann_var  = compute_era5_regression_strict(e5_monthly, season="annual")
    e5_djfm_reg, e5_djfm_var = compute_era5_regression_strict(e5_monthly, season="DJFM")

    # --------------------------------------------------
    # Step 3. Save regression maps
    # --------------------------------------------------
    w4_ann_reg.to_netcdf(WACCM4_ANN_EOF_NC)
    w4_djfm_reg.to_netcdf(WACCM4_DJFM_EOF_NC)
    w6_ann_reg.to_netcdf(WACCM6_ANN_EOF_NC)
    w6_djfm_reg.to_netcdf(WACCM6_DJFM_EOF_NC)
    e5_ann_reg.to_netcdf(ERA5_ANN_EOF_NC)
    e5_djfm_reg.to_netcdf(ERA5_DJFM_EOF_NC)

    print("✅ Saved regression-map netCDF files.")

    # --------------------------------------------------
    # Step 4. Plot 2x3 comparison
    # --------------------------------------------------
    print("\n🎨 Drawing 2x3 comparison figure ...")

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(23, 14),
        subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )

    plot_list = [
        (w4_ann_reg,  w4_ann_var,  "WACCM4 Annual AO", axes[0, 0]),
        (w6_ann_reg,  w6_ann_var,  "WACCM6 Annual AO", axes[0, 1]),
        (e5_ann_reg,  e5_ann_var,  "ERA5 Annual AO",   axes[0, 2]),
        (w4_djfm_reg, w4_djfm_var, "WACCM4 DJFM AO",   axes[1, 0]),
        (w6_djfm_reg, w6_djfm_var, "WACCM6 DJFM AO",   axes[1, 1]),
        (e5_djfm_reg, e5_djfm_var, "ERA5 DJFM AO",     axes[1, 2]),
    ]

    cf_last = None
    for eof_reg, var_pct, ttl, ax in plot_list:
        cf_last = plot_one_panel(ax, eof_reg, var_pct, ttl)

    cbar_ax = fig.add_axes([0.18, 0.06, 0.64, 0.028])
    cbar = fig.colorbar(
        cf_last,
        cax=cbar_ax,
        orientation="horizontal",
        ticks=[-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
    )
    cbar.ax.tick_params(labelsize=11)
    cbar.set_label("1000 mb height regression (m)", fontsize=13)

    plt.subplots_adjust(wspace=0.08, hspace=0.18, bottom=0.13)
    plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
    print(f"✅ Figure saved: {OUT_FIG}")

    plt.show()

    print("\n🎉 All done.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import gc
import re
import glob
import dask
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from multiprocessing import Pool
from dask.diagnostics import ProgressBar
from eofs.xarray import Eof

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

dask.config.set(scheduler="threads", num_workers=8)

# ============================================================
# 1. PATHS / CONFIG
# ============================================================
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

CACHE_DIR = os.path.join(FIG_OUT_DIR, "z1000_monthly_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

WACCM4_BASE_DIR = "/mnt/soclim0/public_data/weiji/B2000WCN001002"
WACCM6_FILE = "/mnt/soclim0/andreas/zg_Amon_CESM2-WACCM_amip_r1i1p1f1_gn_195001-201412.nc"
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"

OUT_FIG = os.path.join(FIG_OUT_DIR, "WACCM4_vs_WACCM6_vs_ERA5_AO_Comparison_DJFM.png")

WACCM4_CACHE = os.path.join(CACHE_DIR, "WACCM4_Z1000_monthly.nc")
WACCM6_CACHE = os.path.join(CACHE_DIR, "WACCM6_Z1000_monthly.nc")
ERA5_CACHE   = os.path.join(CACHE_DIR, "ERA5_Z1000_daily_and_monthly.nc")

WACCM4_ANN_EOF_NC  = os.path.join(CACHE_DIR, "WACCM4_Annual_EOF1_Regression.nc")
WACCM4_DJFM_EOF_NC = os.path.join(CACHE_DIR, "WACCM4_DJFM_EOF1_Regression.nc")
WACCM6_ANN_EOF_NC  = os.path.join(CACHE_DIR, "WACCM6_Annual_EOF1_Regression.nc")
WACCM6_DJFM_EOF_NC = os.path.join(CACHE_DIR, "WACCM6_DJFM_EOF1_Regression.nc")
ERA5_ANN_EOF_NC    = os.path.join(CACHE_DIR, "ERA5_Annual_EOF1_Regression_STRICT.nc")
ERA5_DJFM_EOF_NC   = os.path.join(CACHE_DIR, "ERA5_DJFM_EOF1_Regression_STRICT.nc")

LAT_MIN_AO = 20.0
LAT_MAX_AO = 90.0

ERA5_YEAR_MIN = 1950
ERA5_YEAR_MAX = 2020
ERA5_BASE_START = "1979-01-01"
ERA5_BASE_END   = "2000-12-31"

CORES = 10
SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 104
G = 9.80665

TARGET_PLEV_HPA = np.array([1000.0], dtype=np.float64)
TARGET_PLEV_PA  = TARGET_PLEV_HPA * 100.0

noaa_colors = [
    "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5", "#a3dcfb", "#c2f0fb",
    "#dff8fd", "#f0fcfd", "#ffffff", "#fefbd9", "#feea9e", "#fec762",
    "#f27932", "#b82522"
]
bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
cmap_noaa = mcolors.ListedColormap(noaa_colors)
norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)

# ============================================================
# 2. TOOLS
# ============================================================
def shift_time_cftime(ds, year_offset):
    times = ds["time"].values
    new_times = [t.replace(year=t.year + year_offset) for t in times]
    ds = ds.copy()
    ds["time"] = xr.DataArray(new_times, coords=[new_times], dims=["time"])
    return ds

def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)
    out = np.interp(ltp, lp, z)

    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out

def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )
    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    return z_on_plev

def _extrapolate_to_1000hpa_1d(z_prof, p_prof):
    target_p = 100000.0
    z_prof = np.asarray(z_prof, dtype=np.float64)
    p_prof = np.asarray(p_prof, dtype=np.float64)

    mask = np.isfinite(z_prof) & np.isfinite(p_prof) & (p_prof > 0)
    if not np.any(mask):
        return np.nan

    z = z_prof[mask]
    p = p_prof[mask]

    if len(z) < 2:
        return z[0] if len(z) == 1 else np.nan

    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(target_p)

    if ltp > lp[-1]:
        slope = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        return z[-1] + slope * (ltp - lp[-1])
    elif ltp < lp[0]:
        slope = (z[1] - z[0]) / (lp[1] - lp[0])
        return z[0] + slope * (ltp - lp[0])
    else:
        return np.interp(ltp, lp, z)

def extrapolate_zg_to_1000(zg_da):
    plev_da = zg_da["plev"]
    zg_1000_da = xr.apply_ufunc(
        _extrapolate_to_1000hpa_1d,
        zg_da,
        plev_da,
        input_core_dims=[["plev"], ["plev"]],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[np.float64]
    )
    zg_1000_da.name = "Z1000"
    return zg_1000_da

def select_djfm(da_monthly):
    return da_monthly.sel(time=da_monthly["time.month"].isin([12, 1, 2, 3]))

def plot_one_panel(ax, eof_reg, var_pct, title):
    ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
    ax.coastlines(linewidth=1)
    ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

    eof_cyclic, lon_cyclic = add_cyclic_point(eof_reg.values, coord=eof_reg["lon"])
    cf = ax.contourf(
        lon_cyclic, eof_reg["lat"], eof_cyclic,
        levels=bounds,
        transform=ccrs.PlateCarree(),
        cmap=cmap_noaa,
        norm=norm_noaa,
        extend="both"
    )
    ax.set_title(f"{title}\nVar: {var_pct:.1f}%", fontsize=15, fontweight="bold", pad=14)
    return cf

# ============================================================
# 3. WACCM4 BUILD / LOAD
# ============================================================
def process_waccm4_year(year_int):
    f_z3_name = f"B2000WCN.sample.cam.h3.{year_int:04d}.Z3.nc"
    f_ps_name = f"B2000WCN.sample.cam.h3.{year_int:04d}.PS.nc"

    f_z3 = os.path.join(WACCM4_BASE_DIR, "Z3", f_z3_name)
    if not os.path.exists(f_z3):
        f_z3 = os.path.join(WACCM4_BASE_DIR, f_z3_name)

    f_ps = os.path.join(WACCM4_BASE_DIR, "PS", f_ps_name)
    if not os.path.exists(f_ps):
        f_ps = os.path.join(WACCM4_BASE_DIR, f_ps_name)

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]
            p3d = hyam * p0 + hybm * ps

            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, LAT_MAX_AO))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, LAT_MAX_AO))

            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            da_z3_1000 = z3_plev_nh.sel(lev=1000.0)
            da_z3_1000.name = "Z1000"

            if year_int >= 105:
                da_z3_1000 = shift_time_cftime(
                    da_z3_1000.to_dataset(name="Z1000"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z1000"]

            return da_z3_1000.compute()
    except Exception as e:
        print(f"Error in WACCM4 year {year_int:04d}: {e}")
        return None

def build_waccm4_monthly(cache_path=WACCM4_CACHE, overwrite=False):
    if os.path.exists(cache_path) and not overwrite:
        print(f"📦 Loading cached WACCM4 monthly Z1000: {cache_path}")
        return xr.open_dataset(cache_path, use_cftime=True)["Z1000"]

    print("\n🚀 Building WACCM4 monthly Z1000 from raw files...")
    z3_files = glob.glob(os.path.join(WACCM4_BASE_DIR, "Z3", "*.nc"))
    if not z3_files:
        z3_files = glob.glob(os.path.join(WACCM4_BASE_DIR, "*.Z3.nc"))

    max_year = len(z3_files) if len(z3_files) > 0 else 250
    years_to_process = list(range(1, max_year + 1))

    with Pool(processes=CORES) as pool:
        results = pool.map(process_waccm4_year, years_to_process)

    valid_results = [res for res in results if res is not None]
    z1000_daily_all = xr.concat(valid_results, dim="time").sortby("time")
    z1000_monthly = z1000_daily_all.resample(time="1MS").mean().dropna(dim="time", how="all")
    z1000_monthly.name = "Z1000"
    z1000_monthly.to_dataset(name="Z1000").to_netcdf(cache_path)
    return z1000_monthly

# ============================================================
# 4. WACCM6 BUILD / LOAD
# ============================================================
def build_waccm6_monthly(cache_path=WACCM6_CACHE, overwrite=False):
    if os.path.exists(cache_path) and not overwrite:
        print(f"📦 Loading cached WACCM6 monthly Z1000: {cache_path}")
        return xr.open_dataset(cache_path, use_cftime=True)["Z1000"]

    print("\n🚀 Building WACCM6 monthly Z1000 from raw file...")
    ds_w6 = xr.open_dataset(WACCM6_FILE, use_cftime=True)
    zg = ds_w6["zg"].sel(lat=slice(LAT_MIN_AO, LAT_MAX_AO))

    plev_vals = zg["plev"].values
    if np.nanmax(plev_vals) < 2000:
        zg = zg.assign_coords(plev=zg["plev"] * 100.0)

    with ProgressBar():
        zg_all = zg.compute()
        z1000_monthly = extrapolate_zg_to_1000(zg_all).compute()

    z1000_monthly.name = "Z1000"
    z1000_monthly.to_dataset(name="Z1000").to_netcdf(cache_path)
    return z1000_monthly

# ============================================================
# 5. ERA5 BUILD / LOAD
# ============================================================
def build_era5_daily_monthly(cache_path=ERA5_CACHE, overwrite=False):
    if os.path.exists(cache_path) and not overwrite:
        print(f"📦 Loading cached ERA5 daily/monthly Z1000: {cache_path}")
        ds = xr.open_dataset(cache_path)
        return ds["Z1000_daily"], ds["Z1000_monthly"]

    print("\n🚀 Building ERA5 daily/monthly Z1000 from raw files...")

    pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
    all_files = sorted(glob.glob(os.path.join(ERA5_Z_DIR, "z_4daily*.nc")))
    z_files = [
        f for f in all_files
        if pattern.match(os.path.basename(f))
        and ERA5_YEAR_MIN <= int(pattern.match(os.path.basename(f)).group(1)[:4]) <= ERA5_YEAR_MAX
    ]

    ds = xr.open_mfdataset(z_files, combine="by_coords", parallel=True, chunks={"time": 180})

    print("⏳ 正在截取 ERA5 1000 hPa 区域并加载到内存...")
    with ProgressBar():
        Z_raw = ds["z"].sel(level=1000, latitude=slice(LAT_MAX_AO, LAT_MIN_AO)).compute()

    mean_val = Z_raw.mean().values
    if mean_val > 500:
        Z_1000 = Z_raw / G
    else:
        Z_1000 = Z_raw

    Z_daily = Z_1000.resample(time="1D").mean()
    Z_monthly = Z_daily.resample(time="1MS").mean()

    Z_daily = Z_daily.rename({"latitude": "lat", "longitude": "lon"})
    Z_monthly = Z_monthly.rename({"latitude": "lat", "longitude": "lon"})

    Z_daily.name = "Z1000_daily"
    Z_monthly.name = "Z1000_monthly"

    xr.Dataset({
        "Z1000_daily": Z_daily,
        "Z1000_monthly": Z_monthly
    }).to_netcdf(cache_path)

    return Z_daily, Z_monthly

# ============================================================
# 6. EOF CALC
# ============================================================
def compute_model_monthly_regression(monthly_da, season="annual"):
    # WACCM逻辑保持不动
    if season.upper() == "ANNUAL":
        work = monthly_da
    elif season.upper() == "DJFM":
        work = select_djfm(monthly_da)
    else:
        raise ValueError("season must be 'annual' or 'DJFM'")

    clim = work.groupby("time.month").mean("time")
    anom = (work.groupby("time.month") - clim).drop_vars("month")

    coslat = np.clip(np.cos(np.deg2rad(anom["lat"])), 0, None)
    wgts_2d = np.sqrt(coslat).broadcast_like(
        anom.isel(time=0).drop_vars("time", errors="ignore")
    )

    solver = Eof(anom, weights=wgts_2d.values)
    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    explained_var = float(solver.varianceFraction(neigs=1)[0].values * 100.0)

    if eof1_reg.sel(lat=slice(75, 90)).mean() > 0:
        eof1_reg = eof1_reg * -1.0

    return eof1_reg, explained_var

def compute_era5_regression_strict(era5_daily, season="annual"):
    # 严格照你的 ERA5 validation pipeline 的 regression 图逻辑
    base_daily = era5_daily.sel(time=slice(ERA5_BASE_START, ERA5_BASE_END))

    # 先 monthly，再选 DJFM，避免空桶
    Z_base_monthly = base_daily.resample(time="1MS").mean()

    if season.upper() == "ANNUAL":
        work = Z_base_monthly
    elif season.upper() == "DJFM":
        work = Z_base_monthly.sel(time=Z_base_monthly["time.month"].isin([12, 1, 2, 3]))
    else:
        raise ValueError("season must be 'annual' or 'DJFM'")

    work = work.dropna(dim="time", how="all")

    clim_monthly = work.groupby("time.month").mean("time")
    anom_monthly_base = (work.groupby("time.month") - clim_monthly).drop_vars("month")
    anom_monthly_base = anom_monthly_base.dropna(dim="time", how="all")

    if anom_monthly_base.sizes.get("time", 0) == 0:
        raise RuntimeError("ERA5 DJFM/annual anomaly has zero valid time samples after preprocessing.")

    coslat = np.cos(np.deg2rad(anom_monthly_base["lat"]))
    wgts_2d = np.sqrt(coslat).broadcast_like(anom_monthly_base.isel(time=0))

    solver = Eof(anom_monthly_base, weights=wgts_2d.values)
    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    explained_var = float(solver.varianceFraction(neigs=1)[0].values * 100.0)

    if eof1_reg.sel(lat=slice(90, 75)).mean() > 0:
        eof1_reg = eof1_reg * -1.0

    return eof1_reg, explained_var

# ============================================================
# 7. MAIN
# ============================================================
if __name__ == "__main__":

    # 能读缓存就直接读，不重新算
    w4_monthly = build_waccm4_monthly(cache_path=WACCM4_CACHE, overwrite=False)
    w6_monthly = build_waccm6_monthly(cache_path=WACCM6_CACHE, overwrite=False)
    e5_daily, e5_monthly = build_era5_daily_monthly(cache_path=ERA5_CACHE, overwrite=False)

    print("\n🧮 Computing WACCM4 annual / DJFM regression ...")
    w4_ann_reg,  w4_ann_var  = compute_model_monthly_regression(w4_monthly, season="annual")
    w4_djfm_reg, w4_djfm_var = compute_model_monthly_regression(w4_monthly, season="DJFM")

    print("\n🧮 Computing WACCM6 annual / DJFM regression ...")
    w6_ann_reg,  w6_ann_var  = compute_model_monthly_regression(w6_monthly, season="annual")
    w6_djfm_reg, w6_djfm_var = compute_model_monthly_regression(w6_monthly, season="DJFM")

    print("\n🧮 Computing ERA5 annual / DJFM regression ...")
    e5_ann_reg,  e5_ann_var  = compute_era5_regression_strict(e5_daily, season="annual")
    e5_djfm_reg, e5_djfm_var = compute_era5_regression_strict(e5_daily, season="DJFM")

    w4_ann_reg.to_netcdf(WACCM4_ANN_EOF_NC)
    w4_djfm_reg.to_netcdf(WACCM4_DJFM_EOF_NC)
    w6_ann_reg.to_netcdf(WACCM6_ANN_EOF_NC)
    w6_djfm_reg.to_netcdf(WACCM6_DJFM_EOF_NC)
    e5_ann_reg.to_netcdf(ERA5_ANN_EOF_NC)
    e5_djfm_reg.to_netcdf(ERA5_DJFM_EOF_NC)

    print("\n🎨 Drawing 2x3 comparison figure ...")
    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(23, 14),
        subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )

    plot_list = [
        (w4_ann_reg,  w4_ann_var,  "WACCM4 Annual AO", axes[0, 0]),
        (w6_ann_reg,  w6_ann_var,  "WACCM6 Annual AO", axes[0, 1]),
        (e5_ann_reg,  e5_ann_var,  "ERA5 Annual AO",   axes[0, 2]),
        (w4_djfm_reg, w4_djfm_var, "WACCM4 DJFM AO",   axes[1, 0]),
        (w6_djfm_reg, w6_djfm_var, "WACCM6 DJFM AO",   axes[1, 1]),
        (e5_djfm_reg, e5_djfm_var, "ERA5 DJFM AO",     axes[1, 2]),
    ]

    cf_last = None
    for eof_reg, var_pct, ttl, ax in plot_list:
        cf_last = plot_one_panel(ax, eof_reg, var_pct, ttl)

    cbar_ax = fig.add_axes([0.18, 0.06, 0.64, 0.028])
    cbar = fig.colorbar(
        cf_last,
        cax=cbar_ax,
        orientation="horizontal",
        ticks=[-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
    )
    cbar.set_label("1000 mb height regression (m)", fontsize=13)

    plt.subplots_adjust(wspace=0.08, hspace=0.18, bottom=0.13)
    plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
    print(f"✅ Figure saved: {OUT_FIG}")

    plt.show()